# Validierung Technischer Berechnungsgrundlagen (TB)

Dieses Notebook prüft eine als **PDF** vorliegende TB der privaten
Krankenversicherung automatisiert auf formale und inhaltliche Fehler und
erstellt einen strukturierten Prüfbericht. Es ist bewusst **universell**
gehalten: Die Parser arbeiten mit generischen Mustern (Überschriften,
gedrucktes Inhaltsverzeichnis, Alterstabellen, Formelsatz), sodass auch
andere TB-Dokumente geprüft werden können – Checks, deren Voraussetzungen
ein Dokument nicht erfüllt, werden schlicht übersprungen.

**Prüf-Katalog** (modular erweiterbar, vgl. Abschnitte 5 und 7):

| Bereich | Checks |
|---|---|
| dokumentintern | kaputte/fehlende interne Verweise, Inhaltsverzeichnis, Vollständigkeit des Symbolverzeichnisses, Tabellen-Identitäten und Rechnungszins, Monotonie der Beiträge, Konsistenz benannter Parameter, Versions-/Stichtagskonsistenz, Vollständigkeit der Alterstabellen, Rechtschreibung (LLM), Nachrechnen extrahierter Formeln (LLM), Konsistenz Abschnitt ↔ Anlage (LLM), Vision-Prüfung des Formelsatzes (LLM), Grafik-Tabellen-Abgleich (LLM) |
| dokumentextern | Abgleich mit Referenzwerten (CSV), Abgleich mit der Referenzdatenbank (SQLite), Gesetzes-/Paragrafen-Prüfung, Rahmen-TB-Referenzen, Plausibilität der Normzitate (LLM) |

**Zwei Prüfebenen:**
1. **Deterministische Checks** – reines Python, laufen immer und sind exakt
   reproduzierbar.
2. **LLM-gestützte Checks** – nutzen Claude (konfigurierbar: Haiku 4.5 oder
   Sonnet 5), u. a. um **Formeln aus dem PDF zu extrahieren und in Python
   nachzurechnen**. Sie laufen nur, wenn die Umgebungsvariable
   `ANTHROPIC_API_KEY` gesetzt ist; andernfalls werden sie sauber
   übersprungen.

Das Notebook läuft von oben nach unten ohne manuelle Eingriffe durch;
anzupassen ist allenfalls die folgende Konfigurationszelle.

## 1 | Konfiguration

⚙️ **Diese Zelle ist die einzige, die angepasst werden muss:** Pfad zur
TB, optionaler Referenzdaten-Ordner und das gewünschte Claude-Modell.

In [1]:
from pathlib import Path

# --- Zu prüfendes Dokument -------------------------------------------------
# Pfad zur TB als PDF. Für den Test stehen zwei Dokumente bereit:
#   ../synthetische-tb/TB_VitalPlus_fehlerhaft.pdf  (13 dokumentierte Fehler)
#   ../synthetische-tb/TB_VitalPlus.pdf             (fehlerfreie Fassung)
TB_PDF_PFAD = Path("..") / "synthetische-tb" / "TB_VitalPlus_fehlerhaft.pdf"

# --- Referenzdaten (optional) ----------------------------------------------
# Ordner mit maschinenlesbaren Referenzwerten (CSV; Semikolon-getrennt,
# Dezimalpunkt, eine Spalte "x" für das Alter). Auf None setzen, wenn keine
# Referenzdaten vorliegen – der Abgleich-Check wird dann übersprungen.
REFERENZ_CSV_ORDNER = Path("..") / "synthetische-tb" / "kalkulation" / "out"

# SQLite-Referenzdatenbank mit Sterbe-/Stornowahrscheinlichkeiten und
# Kopfschäden (Tabelle 'rechnungsgrundlagen'). Auf None setzen, wenn keine
# Datenbank vorliegt – der Datenbank-Check wird dann übersprungen.
REFERENZ_DB_PFAD = (Path("..") / "synthetische-tb" / "kalkulation" / "out"
                    / "rechnungsgrundlagen.db")

# --- LLM-Konfiguration -------------------------------------------------------
# Modell für die KI-gestützten Checks. Referenzmodell ist Sonnet 5: Es fand
# in der Validierung (Juli 2026) alle 13 dokumentierten Fehler der
# Testvariante – darunter per Vision-Prüfung ein Formel-Vorzeichen, das im
# PDF-Text unsichtbar ist (L-05), und die überhöhte Kurve der Profil-Grafik
# (L-06) – und prüft nahezu rauschfrei (Details in Abschnitt 7). Haiku 4.5
# (schnell und günstig) eignet sich für Zwischenprüfungen, übersieht aber
# insbesondere Vision-Befunde. Umstellen = eine Zeile tauschen.
MODELL = "claude-sonnet-5"
# MODELL = "claude-haiku-4-5"

MAX_LLM_AUFRUFE = 120                    # Sicherheitsobergrenze je Notebook-Lauf
LLM_CACHE_PFAD = Path("llm_cache.json")  # Antwort-Cache: erneute Läufe kosten nichts

# --- Gesetzestexte (L-04) ----------------------------------------------------
# L-04 schlägt den amtlichen Wortlaut zitierter Vorschriften per Werkzeugaufruf
# live auf gesetze-im-internet.de nach. Ohne Internet (oder mit
# NORMTEXT_QUELLE = None) dient die gespeicherte Kopie als Ersatz;
# NORMTEXT_KOPIE_AKTUALISIEREN = True schreibt live abgerufene Texte dorthin.
NORMTEXT_QUELLE = "https://www.gesetze-im-internet.de"
NORMTEXT_KOPIE = Path("normtexte.json")
NORMTEXT_KOPIE_AKTUALISIEREN = True

# --- Ausgabe -----------------------------------------------------------------
# None = automatisch "pruefbericht_<pdfname>.json" neben dem Notebook.
BERICHT_PFAD = None

## 2 | Setup

Installiert fehlende Pakete bei Bedarf automatisch und prüft, ob das
Prüfobjekt existiert.

In [2]:
import importlib
import subprocess
import sys

# Benötigte Pakete: (Import-Name, pip-Name)
_PAKETE = [("fitz", "pymupdf"), ("pandas", "pandas"), ("anthropic", "anthropic")]

for import_name, pip_name in _PAKETE:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installiere fehlendes Paket: {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pip_name])

import ast
import base64
import hashlib
import json
import math
import os
import re
import sqlite3
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import fitz  # PyMuPDF
import pandas as pd

print("PyMuPDF:", fitz.__version__, "| pandas:", pd.__version__)
print("Prüfobjekt:", TB_PDF_PFAD)
if not Path(TB_PDF_PFAD).exists():
    raise FileNotFoundError(
        f"TB-PDF nicht gefunden: {TB_PDF_PFAD} – bitte TB_PDF_PFAD in der Konfigurationszelle anpassen."
    )

PyMuPDF: 1.27.2.2 | pandas: 3.0.1
Prüfobjekt: ..\2_Synthetische_TB\TB_VitalPlus_fehlerhaft.pdf


## 3 | Prüf-Framework

Alle Checks folgen demselben Muster: Eine Funktion erhält das eingelesene
Dokument und liefert eine Liste von **Befunden** (je mit Beschreibung,
Seite und Schwere `FEHLER`/`WARNUNG`) sowie einen Hinweis zum Prüfumfang.
Der Status des Checks (`BESTANDEN`/`WARNUNG`/`FEHLER`/`ÜBERSPRUNGEN`)
ergibt sich aus den Befunden. Über den Dekorator `@check(...)` registriert
sich jeder Check selbst – **neue Checks lassen sich ergänzen, ohne
bestehenden Code anzufassen.**

Außerdem hier: Hilfsfunktionen für deutsche Zahlformate und die **sichere
Auswertung** von LLM-extrahierten Formeln (AST-basiert, nur Arithmetik –
es wird nie unkontrollierter Code ausgeführt).

In [3]:
# =============================================================================
# Prüf-Framework: Ergebnisstrukturen, Check-Registry und Hilfsfunktionen
# =============================================================================

# --- Ergebnis-Status ---------------------------------------------------------
BESTANDEN = "BESTANDEN"            # Check gelaufen, keine Auffälligkeiten
WARNUNG = "WARNUNG"                # Auffälligkeiten, die geprüft werden sollten
FEHLER = "FEHLER"                  # klare Befunde
UEBERSPRUNGEN = "ÜBERSPRUNGEN"     # Check nicht ausführbar (z. B. kein API-Key)
FEHLGESCHLAGEN = "FEHLGESCHLAGEN"  # Check selbst ist abgestürzt (Programmfehler)


@dataclass
class Befund:
    """Ein einzelner Befund eines Checks."""
    beschreibung: str
    seite: int | None = None       # PDF-Seite (1-basiert), falls lokalisierbar
    schwere: str = FEHLER          # FEHLER oder WARNUNG


@dataclass
class Pruefergebnis:
    """Gesamtergebnis eines Checks."""
    check_id: str
    titel: str
    kategorie: str                 # "dokumentintern" / "dokumentextern"
    methode: str                   # "deterministisch" / "LLM"
    status: str = BESTANDEN
    befunde: list = field(default_factory=list)
    hinweis: str = ""              # Freitext: Umfang, Einschränkungen, Skip-Grund
    dauer_s: float = 0.0


class CheckUebersprungen(Exception):
    """Wird von einem Check geworfen, wenn er nicht ausführbar ist."""


# --- Check-Registry ------------------------------------------------------------
# Jeder Check registriert sich hier selbst; der Runner (weiter unten) führt
# alle registrierten Checks in Reihenfolge aus. Neue Checks lassen sich damit
# ergänzen, ohne bestehenden Code anzufassen (modular erweiterbar).
CHECKS: list = []


def check(check_id: str, titel: str, kategorie: str, methode: str):
    """Dekorator: registriert eine Check-Funktion.

    Die Funktion erhält das eingelesene Dokument (TBDokument) und gibt
    (befunde, hinweis) zurück; der Status wird aus den Befunden abgeleitet.
    """
    def dekorator(fn):
        CHECKS.append({"id": check_id, "titel": titel, "kategorie": kategorie,
                       "methode": methode, "fn": fn})
        return fn
    return dekorator


# --- Text- und Zahl-Hilfsfunktionen -------------------------------------------
def nfkc(s: str) -> str:
    """NFKC-Normalisierung: bildet mathematische Unicode-Buchstaben
    (z. B. 𝑞, 𝐷, 𝜎 aus dem LaTeX-Formelsatz) auf gewöhnliche Zeichen ab."""
    return unicodedata.normalize("NFKC", s)


_ZAHL_RE = re.compile(r"^-?\d{1,3}(?:\.\d{3})*(?:,\d+)?$|^-?\d+(?:,\d+)?$")


def ist_de_zahl(s: str) -> bool:
    """Erkennt deutsche Zahldarstellungen wie '1.234,56' oder '0,037271'."""
    return bool(_ZAHL_RE.match(s.strip()))


def de_zahl(s: str) -> float:
    """Wandelt '1.234,56' in 1234.56 um."""
    return float(s.strip().replace(".", "").replace(",", "."))


def nachkommastellen(s: str) -> int:
    """Anzahl der angezeigten Nachkommastellen einer deutschen Zahl."""
    s = s.strip()
    return len(s.split(",")[1]) if "," in s else 0


def fmt_de(wert: float, dez: int = 4) -> str:
    """Formatiert eine Zahl in deutscher Schreibweise (Komma)."""
    return f"{wert:.{dez}f}".rstrip("0").rstrip(".").replace(".", ",") or "0"


def zeilen_zu_flaeche(text: str) -> str:
    """Fügt die Zeilen einer Seite zu einer durchsuchbaren Textfläche zusammen
    und macht die Silbentrennung rückgängig ('Höchstrechnungs-\\nzins' ->
    'Höchstrechnungszins'; echte Bindestriche wie 'Rahmen-\\nTB' bleiben)."""
    text = re.sub(r"-\n([a-zäöüß])", r"\1", text)
    text = re.sub(r"-\n([A-ZÄÖÜ0-9])", r"-\1", text)
    return re.sub(r"\s+", " ", text)


# --- Sichere Formel-Auswertung ---------------------------------------------------
# Vom LLM extrahierte Formeln werden als Python-Ausdrücke geliefert und hier
# ausgewertet. Damit kein beliebiger Code ausgeführt werden kann, wird der
# Ausdruck vorab per AST auf harmlose Bausteine (Arithmetik, wenige
# Mathe-Funktionen) geprüft.
_ERLAUBTE_FUNKTIONEN = {"abs": abs, "min": min, "max": max, "round": round,
                        "sqrt": math.sqrt, "exp": math.exp, "log": math.log}
_ERLAUBTE_KNOTEN = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Name,
                    ast.Load, ast.Call, ast.Add, ast.Sub, ast.Mult, ast.Div,
                    ast.Pow, ast.USub, ast.UAdd, ast.Mod, ast.FloorDiv)


def sichere_formelauswertung(ausdruck: str, variablen: dict) -> float:
    """Wertet einen arithmetischen Python-Ausdruck mit den übergebenen
    Variablen aus; alles andere (Attribute, Imports, ...) wird abgelehnt."""
    baum = ast.parse(ausdruck, mode="eval")
    for knoten in ast.walk(baum):
        if not isinstance(knoten, _ERLAUBTE_KNOTEN):
            raise ValueError(f"Unerlaubtes Element in Formel: {type(knoten).__name__}")
        if isinstance(knoten, ast.Call):
            if not (isinstance(knoten.func, ast.Name) and knoten.func.id in _ERLAUBTE_FUNKTIONEN):
                raise ValueError("Unerlaubter Funktionsaufruf in Formel")
    namensraum = dict(_ERLAUBTE_FUNKTIONEN)
    namensraum.update(variablen)
    return float(eval(compile(baum, "<formel>", "eval"), {"__builtins__": {}}, namensraum))

## 4 | PDF-Ingestion: universeller Struktur-Parser

Aus dem PDF wird eine strukturierte Sicht (`TBDokument`) gewonnen:

- **Seiten** mit normalisiertem Text (NFKC bildet die Unicode-Mathebuchstaben
  des Formelsatzes auf normale Zeichen ab, z. B. `𝐷𝑄𝑥 → DQx`) und einer
  „Textfläche“, in der Zeilenumbrüche und Silbentrennungen aufgehoben sind.
- **Überschriften** (`Abschnitt/Anlage/Anhang N.N: Titel`) – ergänzt um die
  vollständigen Titel aus den PDF-Bookmarks, falls vorhanden.
- Das **gedruckte Inhaltsverzeichnis** mit Titeln und Seitenzahlen.
- **Alterstabellen**: koordinatenbasiert erkannte numerische Blöcke; die
  Spaltenköpfe werden über ihre x-Position den Spalten zugeordnet und auf
  kanonische Namen (`q`, `w`, `qw`, `DQ`, `A`, `a`, `P`, `b`, …) abgebildet;
  Fortsetzungen über Seiten hinweg werden zusammengeführt.
- **Gleichungsnummern**, **Normzitate**, **Zins-Deklarationen**,
  **Rahmen-TB-Referenzen** und die im **Symbolverzeichnis** erklärten
  Symbole.

In [4]:
# =============================================================================
# PDF-Ingestion: universeller Struktur-Parser für TB-Dokumente
# =============================================================================
# Ziel: Aus dem PDF wird eine strukturierte Repräsentation gewonnen, auf der
# alle Checks aufsetzen. Die Parser sind bewusst generisch gehalten
# (Überschriften-Muster, gedrucktes Inhaltsverzeichnis, koordinatenbasierte
# Tabellenerkennung), damit auch andere TB-Dokumente verarbeitet werden können.
# Was nicht erkannt wird, bleibt leer – die Checks überspringen dann sauber.

_TEIL_PRAEFIXE = r"(?:Abschnitt|Anlage|Anhang|Teil|Kapitel)"
_GESCHLECHT_LABELS = {"Männer": "M", "Frauen": "F", "Kinder": "K",
                      "Jugendliche": "K", "Kinder/Jugendliche": "K"}

# Kanonische Spaltennamen: bildet die im PDF-Textlayer erscheinenden
# Spaltenköpfe (nach NFKC-Normalisierung, z. B. "DQx" aus $DQ_x$) auf
# einheitliche Kürzel ab. Unbekannte Spalten behalten ihren Rohnamen.
_SPALTEN_KANON = {
    "x": "x", "qx": "q", "wx": "w", "(q+w)x": "qw", "DQx": "DQ",
    "kx": "k", "Kx": "K", "Ax": "A", "ax": "a", "Px": "P", "bx": "b",
    "lx": "l", "Dx": "D", "Ax+1/2": "A12", "ax+1/2": "a12",
}
# Entsprechung für CSV-Referenzdateien (Spaltenköpfe wie "q_x", "A_x+1/2")
_CSV_KANON = {
    "x": "x", "q_x": "q", "w_x": "w", "s_x": "qw", "DQ_x": "DQ",
    "k_x": "k", "K_x": "K", "A_x": "A", "a_x": "a", "P_x": "P", "b_x": "b",
    "A_x+1/2": "A12", "a_x+1/2": "a12",
}


def kanon_spalte(roh: str):
    """Normalisiert einen Spaltenkopf; erkennt Geschlechts-Suffixe (PxM -> P, M)."""
    s = re.sub(r"\s+", "", nfkc(roh))
    s = re.sub(r"\((?:EUR|in|%|p\.a\.)[^)]*\)", "", s).strip(":;,")
    geschlecht = None
    m = re.match(r"^(.*)([MFW])$", s)          # Suffix: "PxM"
    if m and m.group(1) in _SPALTEN_KANON:
        s = m.group(1)
        geschlecht = "F" if m.group(2) in ("F", "W") else "M"
    m = re.match(r"^([MFW])(.+)$", s)          # Präfix: "MAx+1/2" (Superskript-Zeile)
    if m and m.group(2) in _SPALTEN_KANON:
        s = m.group(2)
        geschlecht = "F" if m.group(1) in ("F", "W") else "M"
    m = re.match(r"^(.*?)(Männer|Frauen)$", s)  # Wort-Suffix: "kx Männer"
    if m and m.group(1) in _SPALTEN_KANON:
        s = m.group(1)
        geschlecht = "M" if m.group(2) == "Männer" else "F"
    return _SPALTEN_KANON.get(s, s), geschlecht


@dataclass
class Ueberschrift:
    praefix: str          # "Abschnitt" / "Anlage" / "Anhang"
    nummer: str           # z. B. "2.3.3"
    titel: str            # vollständiger Titel (aus Text, ergänzt um Bookmark)
    seite: int            # PDF-Seite (1-basiert)


@dataclass
class IhvEintrag:
    praefix: str
    nummer: str
    titel: str
    seite_lt_ihv: int     # im IHV abgedruckte (logische) Seitenzahl
    pdf_seite: int        # PDF-Seite, auf der der Eintrag steht


@dataclass
class Tabelle:
    teil: str                       # Kopfzeilen-Teil der Seite(n), z. B. "Anlage 2.2"
    geschlecht: str | None          # "M"/"F"/"K" aus der Tabellenüberschrift
    spalten: list                   # kanonische Spaltennamen (Index 0 = Altersspalte)
    geschlecht_je_spalte: list      # Geschlechts-Suffix je Spalte (z. B. bei "PxM")
    caption: str
    zins_prozent: float | None      # aus der Caption, z. B. "(Rechnungszins 2,7 %)"
    seiten: list
    zeilen: dict                    # Alter (int) -> (Rohwerte als Strings, PDF-Seite)

    def spalten_indizes(self, kanon_name: str):
        """Alle Spaltenindizes mit diesem kanonischen Namen (Duplikate möglich,
        z. B. 'P b P b' für Männer/Frauen nebeneinander)."""
        return [i for i, s in enumerate(self.spalten) if s == kanon_name]

    def wert(self, alter: int, spalten_idx: int):
        werte, _ = self.zeilen[alter]
        return de_zahl(werte[spalten_idx])

    def roh(self, alter: int, spalten_idx: int):
        werte, _ = self.zeilen[alter]
        return werte[spalten_idx]

    def seite_von(self, alter: int):
        return self.zeilen[alter][1]

    def kurzname(self):
        c = re.sub(r"\s*\(Fortsetzung\)\s*", "", self.caption)
        c = re.sub(r"^Tabelle \d+:\s*", "", c).strip()
        kurz = c[:48] + ("…" if len(c) > 48 else "")
        return f"{self.teil or '?'} – {kurz}" if c else (self.teil or "?")


class TBDokument:
    """Strukturierte Sicht auf ein TB-PDF."""

    def __init__(self, pfad):
        self.pfad = Path(pfad)
        self._doc = fitz.open(str(pfad))
        self.seitenzahl = len(self._doc)
        self._seiten_einlesen()
        self._ihv_parsen()
        self._ueberschriften_parsen()
        self._gleichungen_parsen()
        self._tabellen_parsen()
        self._symbole_parsen()
        self._paragrafen_parsen()
        self._zins_deklarationen_parsen()
        self._rahmen_tb_parsen()

    # ------------------------------------------------------------------ Seiten
    def _seiten_einlesen(self):
        self.seiten = []
        for i in range(self.seitenzahl):
            roh = self._doc[i].get_text("text")
            text = nfkc(roh)
            zeilen = text.splitlines()
            teil = ""
            for j, zeile in enumerate(zeilen[:15]):
                if zeile.strip() == "Teil:" and j + 1 < len(zeilen):
                    teil = zeilen[j + 1].strip()
                    break
            self.seiten.append({"nummer": i + 1, "roh": roh, "text": text,
                                "zeilen": zeilen, "teil": teil, "ist_ihv": False,
                                "flaeche": zeilen_zu_flaeche(text)})

    # ------------------------------------------------- gedrucktes Inhaltsverzeichnis
    def _ihv_parsen(self):
        """Erkennt das gedruckte IHV an Zeilen der Form 'Abschnitt 2.1:' ohne
        nachfolgenden Titel in derselben Zeile (Titel und Seitenzahl folgen in
        eigenen Zeilen)."""
        self.ihv = []
        muster = re.compile(rf"^({_TEIL_PRAEFIXE})\s+(\d+(?:\.\d+)*):$")
        for seite in self.seiten:
            treffer = [m for z in seite["zeilen"] if (m := muster.match(z.strip()))]
            if len(treffer) < 3:
                continue
            seite["ist_ihv"] = True
            zeilen = seite["zeilen"]
            i = 0
            while i < len(zeilen):
                m = muster.match(zeilen[i].strip())
                if not m:
                    i += 1
                    continue
                titel_teile, seitenzahl = [], None
                j = i + 1
                while j < len(zeilen) and j <= i + 6:
                    z = zeilen[j].strip()
                    if re.fullmatch(r"\d{1,4}", z):
                        seitenzahl = int(z)
                        break
                    titel_teile.append(z)
                    j += 1
                if seitenzahl is not None:
                    titel = re.sub(r"(?:\s*\.\s*){3,}", " ", " ".join(titel_teile))
                    titel = re.sub(r"[\s.]+$", "", titel).strip()
                    self.ihv.append(IhvEintrag(m.group(1), m.group(2), titel,
                                               seitenzahl, seite["nummer"]))
                i = j + 1

    # ------------------------------------------------------------- Überschriften
    def _ueberschriften_parsen(self):
        """Überschriften im Fließtext ('Anlage 3.1: Titel ...'), ergänzt um die
        vollständigen Titel aus den PDF-Bookmarks (falls vorhanden)."""
        muster = re.compile(rf"^({_TEIL_PRAEFIXE})\s+(\d+(?:\.\d+)*):\s+(\S.*)$")
        bookmarks = defaultdict(list)  # PDF-Seite -> [Titel]
        for _, titel, seite in self._doc.get_toc():
            bookmarks[seite].append(nfkc(titel).strip())
        self.ueberschriften = []
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            for zeile in seite["zeilen"]:
                m = muster.match(zeile.strip())
                if not m:
                    continue
                titel = m.group(3).strip()
                # Bookmark derselben Seite mit passendem Anfang liefert den
                # vollständigen (ggf. umbrochenen) Titel.
                for bm in bookmarks.get(seite["nummer"], []):
                    if bm.startswith(titel[: max(10, len(titel) - 2)]) or titel.startswith(bm):
                        titel = bm
                        break
                self.ueberschriften.append(
                    Ueberschrift(m.group(1), m.group(2), titel, seite["nummer"]))

    def ueberschrift(self, praefix: str, nummer: str):
        for u in self.ueberschriften:
            if u.praefix == praefix and u.nummer == nummer:
                return u
        return None

    # ------------------------------------------------------------- Gleichungen
    def _gleichungen_parsen(self):
        """Definierte Gleichungsnummern ('(23)' am Zeilenende) und Verweise
        darauf ('Gleichung (23)', 'Gl. (19)', 'Rekursionen (20)/(21)')."""
        self.gleichungs_defs = set()
        self.gleichungs_refs = []  # (nummer, seite)
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            for zeile in seite["zeilen"]:
                m = re.search(r"\((\d{1,3})\)\s*$", zeile)
                if m:
                    self.gleichungs_defs.add(int(m.group(1)))
            flaeche = seite["flaeche"]
            for m in re.finditer(
                    r"(?:Gleichung(?:en)?|Gl\.|Rekursion(?:en)?)\s*((?:\(\d{1,3}\)[\s/,]*(?:und\s*)?)+)",
                    flaeche):
                for n in re.findall(r"\((\d{1,3})\)", m.group(1)):
                    self.gleichungs_refs.append((int(n), seite["nummer"]))

    # ----------------------------------------------------------------- Tabellen
    _STOPPWOERTER = {"und", "oder", "der", "die", "das", "den", "dem", "des", "ein",
                     "eine", "für", "mit", "nach", "aus", "bei", "von", "vom", "zum",
                     "zur", "im", "in", "am", "an", "auf", "ist", "sind", "wird",
                     "werden", "wurden", "sie", "es", "gemäß", "sowie", "als", "wie",
                     "dieser", "diese", "dieses", "gegenüber"}

    def _zeilen_aus_words(self, seiten_idx: int, tol: float = 2.0):
        """Gruppiert die Wörter einer Seite anhand ihrer y-Koordinate zu
        visuellen Zeilen (robust gegenüber Hoch-/Tiefstellungen).
        Rückgabe je Zeile: Liste von (x0, x1, Text), nach x sortiert."""
        zeilen = []
        for x0, y0, x1, y1, txt, *_ in self._doc[seiten_idx].get_text("words"):
            t = nfkc(txt)
            for z in zeilen:
                if abs(z[0] - y0) <= tol:
                    z[1].append((x0, x1, t))
                    break
            else:
                zeilen.append([y0, [(x0, x1, t)]])
        zeilen.sort(key=lambda z: z[0])
        return [sorted(toks) for _, toks in zeilen]

    @staticmethod
    def _tokens_kleben(toks, max_luecke: float = 1.5):
        """Fügt horizontal direkt aneinanderstoßende Tokens zusammen
        (z. B. 'Ax'+'M' aus hochgestellten Indizes, '(q+'+'w)x')."""
        ergebnis = []
        for x0, x1, t in toks:
            if ergebnis and x0 - ergebnis[-1][1] < max_luecke:
                ergebnis[-1] = (ergebnis[-1][0], x1, ergebnis[-1][2] + t)
            else:
                ergebnis.append((x0, x1, t))
        return ergebnis

    def _tabellen_parsen(self):
        """Erkennt Alterstabellen: zusammenhängende Blöcke rein numerischer
        Zeilen, deren erste Spalte ein Alter (0..130) ist. Spaltenköpfe werden
        den Spalten über x-Überlappung zugeordnet; Fortsetzungen über mehrere
        Seiten werden zu einer logischen Tabelle zusammengeführt."""
        fragmente = []
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            zeilen = self._zeilen_aus_words(seite["nummer"] - 1)
            daten = []
            for toks in zeilen:
                werte = [t for _, _, t in toks]
                ist_daten = (len(werte) >= 3 and all(ist_de_zahl(t) for t in werte)
                             and de_zahl(werte[0]).is_integer()
                             and 0 <= de_zahl(werte[0]) <= 130)
                daten.append(ist_daten)
            i = 0
            while i < len(zeilen):
                if not daten[i]:
                    i += 1
                    continue
                n_spalten = len(zeilen[i])
                j = i
                while j < len(zeilen) and daten[j] and len(zeilen[j]) == n_spalten:
                    j += 1
                fragmente.append(self._fragment_bauen(seite, zeilen, i, j, n_spalten))
                i = j
        self._fragmente_zusammenfuehren(fragmente)

    def _fragment_bauen(self, seite, zeilen, start, ende, n_spalten):
        block = zeilen[start:ende]
        # Spaltenintervalle [min x0, max x1] aus den Datenzeilen
        intervalle = []
        for s in range(n_spalten):
            intervalle.append((min(z[s][0] for z in block), max(z[s][1] for z in block)))

        def ueberlappt(tok, toleranz=6.0):
            for s, (lo, hi) in enumerate(intervalle):
                if tok[0] < hi + toleranz and tok[1] > lo - toleranz:
                    return s
            return None

        # Phase A – Kopfzeilen: bis zu 2 spaltenartige Zeilen direkt oberhalb.
        # Fließtext wird über Stoppwörter, Tokenlänge und fehlende
        # Spaltenüberlappung aussortiert.
        koepfe = [[] for _ in range(n_spalten)]
        k = start - 1
        kopf_zeilen = 0
        while k >= 0 and kopf_zeilen < 2:
            toks = self._tokens_kleben(zeilen[k])
            werte = [t for _, _, t in toks]
            zeilen_text = " ".join(werte)
            if re.search(r"^Tabelle \d+|Fortsetzung|Rechnungszins", zeilen_text):
                break  # Caption erreicht -> Phase B übernimmt diese Zeile
            if not werte or len(werte) > n_spalten + 4 \
                    or (len(werte) == 1 and len(werte[0]) > 4) \
                    or any(w.lower() in self._STOPPWOERTER for w in werte) \
                    or any(len(w) > 14 for w in werte):
                break
            treffer = [(tok, ueberlappt(tok)) for tok in toks]
            if sum(1 for _, s in treffer if s is None) > len(toks) * 0.3:
                break
            zeile_koepfe = [[] for _ in range(n_spalten)]
            for (x0, x1, t), s in treffer:
                if s is not None:
                    zeile_koepfe[s].append(t)
            for s in range(n_spalten):
                koepfe[s] = zeile_koepfe[s] + koepfe[s]
            kopf_zeilen += 1
            k -= 1
        # Phase B – Caption: Zeilen mit 'Tabelle N:', '(Fortsetzung)' oder
        # '(Rechnungszins ...)' direkt oberhalb der Kopfzeilen.
        caption_teile = []
        while k >= 0 and len(caption_teile) < 4:
            text = " ".join(t for _, _, t in zeilen[k])
            if re.search(r"^Tabelle \d+|Fortsetzung|Rechnungszins", text):
                caption_teile.insert(0, text)
                k -= 1
            else:
                break
        caption = " ".join(caption_teile)
        roh_spalten = [" ".join(teile) for teile in koepfe]
        spalten, geschlecht_sp = [], []
        for name in roh_spalten:
            kan, ges = kanon_spalte(name)
            spalten.append(kan)
            geschlecht_sp.append(ges)
        # Geschlecht aus der Caption; nennt sie mehrere, entscheidet das
        # Spalten-Suffix (z. B. PxM/PxF) statt der Tabellen-Ebene.
        gefunden = {kurz for label, kurz in _GESCHLECHT_LABELS.items() if label in caption}
        geschlecht = gefunden.pop() if len(gefunden) == 1 else None
        m = re.search(r"Rechnungszins\s+(\d+,\d+)\s*%", caption)
        zins = de_zahl(m.group(1)) if m else None
        zeilen_dict = {}
        for z in block:
            werte = [t for _, _, t in z]
            zeilen_dict[int(de_zahl(werte[0]))] = (werte, seite["nummer"])
        return {"teil": seite["teil"], "seite": seite["nummer"], "caption": caption,
                "spalten": spalten, "roh_spalten": roh_spalten,
                "geschlecht": geschlecht, "geschlecht_sp": geschlecht_sp,
                "zins": zins, "zeilen": zeilen_dict}

    def _fragmente_zusammenfuehren(self, fragmente):
        self.tabellen = []
        index = {}
        for f in fragmente:
            sig = tuple(f["spalten"])
            t = index.get((f["teil"], f["geschlecht"], sig))
            if t is None and f["geschlecht"] is None:
                # Fortsetzungsseite ohne eigene Caption: an die letzte Tabelle
                # gleicher Struktur im selben Teil anhängen.
                for kandidat in reversed(self.tabellen):
                    if kandidat.teil == f["teil"] and tuple(kandidat.spalten) == sig:
                        t = kandidat
                        break
            if t is not None:
                t.zeilen.update(f["zeilen"])
                if f["seite"] not in t.seiten:
                    t.seiten.append(f["seite"])
                t.zins_prozent = t.zins_prozent or f["zins"]
                t.caption = t.caption or f["caption"]
                if all(g is None for g in t.geschlecht_je_spalte) and any(f["geschlecht_sp"]):
                    t.geschlecht_je_spalte = f["geschlecht_sp"]
            else:
                if len(f["zeilen"]) < 3 and not f["caption"]:
                    continue  # Mini-Blöcke ohne Kontext sind meist keine Tabellen
                t = Tabelle(teil=f["teil"], geschlecht=f["geschlecht"],
                            spalten=f["spalten"], geschlecht_je_spalte=f["geschlecht_sp"],
                            caption=f["caption"], zins_prozent=f["zins"],
                            seiten=[f["seite"]], zeilen=dict(f["zeilen"]))
                index[(f["teil"], f["geschlecht"], sig)] = t
                self.tabellen.append(t)

    # ------------------------------------------------------------------ Symbole
    _MATH_RUN = re.compile(r"[\U0001D400-\U0001D7FF][\U0001D400-\U0001D7FF0-9+/]*")

    def _symbole_parsen(self):
        """Mathematische Symbole: Vorkommen im Formelsatz (Unicode-Mathe-
        Buchstaben) dokumentweit sowie die im Symbolverzeichnis erklärten."""
        # Seitenbereich des Symbolverzeichnisses bestimmen
        glossar_seiten = set()
        kandidaten = [u for u in self.ueberschriften
                      if re.search(r"symbolverzeichnis|glossar|abkürzungen", u.titel, re.I)]
        if kandidaten:
            start = kandidaten[0].seite
            folgende = sorted(u.seite for u in self.ueberschriften if u.seite > start)
            ende = folgende[0] - 1 if folgende else start
            glossar_seiten = set(range(start, ende + 1))
        self.glossar_seiten = glossar_seiten
        self.glossar_symbole = set()
        self.symbol_vorkommen = defaultdict(list)  # Token -> [Seiten]
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            for m in self._MATH_RUN.finditer(seite["roh"]):
                token = nfkc(m.group(0))
                if seite["nummer"] in glossar_seiten:
                    self.glossar_symbole.add(token)
                else:
                    self.symbol_vorkommen[token].append(seite["nummer"])

    # ---------------------------------------------------------------- Paragrafen
    def _paragrafen_parsen(self):
        """Alle Normzitate ('§ 4 KVAV', '§ 146 Abs. 2 VAG', ...) mit Kontext."""
        muster = re.compile(
            r"§§?\s*(\d+[a-z]?)\s*"                              # Paragraf
            r"((?:Abs\.\s*\d+(?:\s*(?:und|bis|[–-])\s*\d+)?\s*)?"  # Absatz/-bereich
            r"(?:Satz\s*\d+\s*)?"                                # Satz
            r"(?:Nrn?\.\s*\d+(?:\s*(?:,|und|bis)\s*\d+)*\s*)?)"   # Nr./Nrn.-Aufzählung
            r"(?!Abs\b|Nrn?\b|Satz\b)"                           # kein Zusatz als Norm
            r"([A-ZÄÖÜ][A-Za-zÄÖÜäöü]*(?:\s+[IVX]+\b)?)")         # Norm (+ SGB-Buch)
        self.paragrafen = []  # dicts: nummer, zusatz, norm, kontext, seite
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            flaeche = seite["flaeche"]
            for m in muster.finditer(flaeche):
                kontext = flaeche[max(0, m.start() - 180):m.end() + 120]
                self.paragrafen.append({
                    "zitat": re.sub(r"\s+", " ", m.group(0)).strip(),
                    "nummer": m.group(1), "norm": m.group(3),
                    "kontext": kontext, "seite": seite["nummer"]})

    # ---------------------------------------------------- Zins-Deklarationen
    def _zins_deklarationen_parsen(self):
        """Sucht Label-/Prozentwert-Tabellen wie 'Männer | Frauen' über
        '2,7 % | 2,0 %' (typisch für die Zins-Anlage einer TB)."""
        self.zins_deklarationen = []  # (label, prozent, seite, teil)
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            zeilen = self._zeilen_aus_words(seite["nummer"] - 1)
            for a in range(len(zeilen) - 1):
                labels = [t for _, _, t in zeilen[a]]
                if not labels or not all(l in _GESCHLECHT_LABELS for l in labels):
                    continue
                werte_toks = [t for _, _, t in zeilen[a + 1]]
                prozente = re.findall(r"(\d+,\d+)\s*%", " ".join(werte_toks))
                if len(prozente) == len(labels):
                    for label, p in zip(labels, prozente):
                        self.zins_deklarationen.append(
                            (label, de_zahl(p), seite["nummer"], seite["teil"]))

    # ------------------------------------------------------------ Rahmen-TB
    def _rahmen_tb_parsen(self):
        """Referenzen auf Rahmen-Berechnungsgrundlagen (z. B. 'RTB-AUZ 2025-1')."""
        self.rahmen_tb_register = {}   # Name -> (Version, Seite) aus dem Register
        self.rahmen_tb_referenzen = []  # (Name, Version|None, Seite)
        register_seiten = set()
        for u in self.ueberschriften:
            if re.search(r"rahmen", u.titel, re.I):
                register_seiten.add(u.seite)
        muster = re.compile(r"\b(RTB-[A-Za-zÄÖÜ0-9]+)\b"
                            r"(?:\s*\(?(?:Version\s*)?(\d{4}-\d+|\d+\.\d+)\)?)?")
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            flaeche = seite["flaeche"]
            for m in muster.finditer(flaeche):
                name, version = m.group(1), m.group(2)
                if seite["nummer"] in register_seiten:
                    if name not in self.rahmen_tb_register or version:
                        self.rahmen_tb_register[name] = (version, seite["nummer"])
                else:
                    self.rahmen_tb_referenzen.append((name, version, seite["nummer"]))

    # -------------------------------------------------------------- Vision
    def grafik_seiten(self, min_pfade: int = 40):
        """Seiten mit eingebetteten Diagrammen: Vektorgrafiken (Plots)
        bestehen aus vielen Zeichenpfaden, Tabellenlinien nur aus wenigen."""
        ergebnis = []
        for seite in self.seiten:
            if seite["ist_ihv"]:
                continue
            if len(self._doc[seite["nummer"] - 1].get_drawings()) >= min_pfade:
                ergebnis.append(seite["nummer"])
        return ergebnis

    def seite_als_png_b64(self, nummer: int, dpi: int = 120) -> str:
        """Rendert eine Seite als PNG (Base64) – Grundlage der Vision-Analyse,
        die Formeln im Originalsatz statt im zerlegten Text-Layer sieht."""
        pix = self._doc[nummer - 1].get_pixmap(dpi=dpi)
        return base64.b64encode(pix.tobytes("png")).decode("ascii")

    # ------------------------------------------------------------- Übersicht
    def zusammenfassung(self):
        print(f"Dokument:        {self.pfad.name} ({self.seitenzahl} Seiten)")
        print(f"IHV-Einträge:    {len(self.ihv)}")
        print(f"Überschriften:   {len(self.ueberschriften)}")
        print(f"Tabellen:        {len(self.tabellen)} (zusammengeführt)")
        for t in self.tabellen:
            print(f"   - {t.teil or '?':14s} G={t.geschlecht or '-'} "
                  f"Spalten={t.spalten} Zeilen={len(t.zeilen)} Seiten={t.seiten}")
        print(f"Gleichungen:     {len(self.gleichungs_defs)} definiert, "
              f"{len(self.gleichungs_refs)} Verweise")
        print(f"Symbole:         {len(self.glossar_symbole)} im Glossar, "
              f"{len(self.symbol_vorkommen)} verschiedene im Text")
        print(f"Normzitate:      {len(self.paragrafen)}")
        print(f"Zins-Angaben:    {self.zins_deklarationen}")
        print(f"Rahmen-TB:       Register={self.rahmen_tb_register}")


dok = TBDokument(TB_PDF_PFAD)
dok.zusammenfassung()

Dokument:        TB_VitalPlus_fehlerhaft.pdf (93 Seiten)
IHV-Einträge:    65
Überschriften:   65
Tabellen:        11 (zusammengeführt)
   - Anlage 2.2     G=M Spalten=['x', 'q', 'w', 'qw', 'DQ'] Zeilen=90 Seiten=[61, 62, 63]
   - Anlage 2.2     G=F Spalten=['x', 'q', 'w', 'qw', 'DQ'] Zeilen=90 Seiten=[64, 65]
   - Anlage 2.3.2   G=- Spalten=['x', 'ambulant', 'stationär', 'Zahn-Beh.', 'Zahn-Ers.', '∑︀ux', 'SummeKbeobx'] Zeilen=101 Seiten=[67, 68, 69]
   - Anlage 2.3.2   G=- Spalten=['x', 'ambulant', 'stationär', 'Zahn-Beh.', 'Zahn-Ers.', '∑︀KSx', '∑︀ux', 'SummeKbeobx'] Zeilen=101 Seiten=[70, 71, 72]
   - Anlage 2.3.3   G=- Spalten=['x', 'k', 'k'] Zeilen=90 Seiten=[73, 74, 75]
   - Anlage 2.3.3   G=- Spalten=['x', 'K', 'K'] Zeilen=90 Seiten=[76, 77]
   - Anlage 3.1     G=- Spalten=['x', 'A', 'a', 'A', 'a'] Zeilen=90 Seiten=[81, 82]
   - Anlage 5.2     G=- Spalten=['x', 'A12', 'a12', 'A12', 'a12'] Zeilen=90 Seiten=[84, 85]
   - Anlage 5.2     G=- Spalten=['xe', 'mMänner', 'mFrauen'] Zeile

## 5 | Deterministische Checks

| Check | prüft |
|---|---|
| V-01 | nicht aufgelöste Verweise („??“) |
| V-02 | Existenz aller referenzierten Abschnitte/Anlagen/Anhänge und Gleichungen |
| V-03 | Inhaltsverzeichnis: Titelgleichheit und Seitenzahlen (mit automatischem Seitenversatz) |
| V-04 | Vollständigkeit des Symbolverzeichnisses (verwendete vs. erklärte Symbole) |
| V-05 | Tabellen-Identitäten: (q+w)ₓ = qₓ + wₓ; implizierter Rechnungszins aus DQₓ vs. deklarierter Zins |
| V-06 | Monotonie der Beitrags-/Nettobeitragsspalten: Rückgänge → Warnung, außer die TB begründet den Rückgang für genau diesen Übergang selbst (z. B. Wegfall von Zuschlägen ab Alter 65) und er bleibt klein (≤ 3 %) |
| V-07 | Zellgenauer Abgleich aller zuordenbaren Tabellenspalten mit den Referenz-CSVs |
| V-08 | Konsistenz benannter Parameter (z. B. G^M) über alle Fundstellen |
| V-09 | Normzitate gegen eine aktuarielle Wissensbasis; Mindesthöhe des Sicherheitszuschlags (§ 7 KVAV) |
| V-10 | Rahmen-TB-Referenzen gegen das Rahmen-TB-Register (inkl. Versionen) |
| V-11 | Versions-/Stichtagskonsistenz: Änderungsnummer und Gültigkeitsdatum überall gleich, Änderungshistorie auf aktuellem Stand |
| V-12 | Vollständigkeit der Alterstabellen: keine Alterslücken, deklariertes Endalter erreicht, plausible Wertebereiche |
| V-13 | Zellgenauer Abgleich der Sterbe-/Stornowahrscheinlichkeiten und Kopfschäden mit der SQLite-Referenzdatenbank (inkl. abgeleiteter Summe (q+w)ₓ) |

In [5]:
# =============================================================================
# Deterministische Checks (laufen immer, auch ohne API-Key)
# =============================================================================

# ---------------------------------------------------------------- V-01
@check("V-01", "Kaputte Verweise („??“)", "dokumentintern", "deterministisch")
def check_kaputte_verweise(dok):
    """LaTeX ersetzt nicht auflösbare Verweise durch '??' – ein sicheres
    Zeichen für einen toten internen Verweis."""
    befunde = []
    for seite in dok.seiten:
        if seite["ist_ihv"]:
            continue
        for m in re.finditer(r"\?\?", seite["text"]):
            kontext = re.sub(r"\s+", " ", seite["text"][max(0, m.start() - 90):m.end() + 40])
            befunde.append(Befund(f"Nicht aufgelöster Verweis: „...{kontext}...“",
                                  seite["nummer"]))
    return befunde, f"{dok.seitenzahl} Seiten durchsucht."


# ---------------------------------------------------------------- V-02
@check("V-02", "Existenz interner Verweise", "dokumentintern", "deterministisch")
def check_interne_verweise(dok):
    """Jeder Verweis auf 'Abschnitt/Anlage/Anhang N.N' und jede referenzierte
    Gleichungsnummer muss im Dokument existieren."""
    befunde = []
    vorhandene = defaultdict(set)
    for u in dok.ueberschriften:
        vorhandene[u.praefix].add(u.nummer)
    ref_muster = re.compile(
        r"\b(Abschnitt(?:s|e|en)?|Anlagen?|Anhangs?|Anhänge)\s+(\d+(?:\.\d+)*)"
        r"((?:\s*[,/]\s*\d+(?:\.\d+)*)*(?:\s*(?:und|bis)\s*\d+(?:\.\d+)*)?)")
    n_refs = 0
    for seite in dok.seiten:
        if seite["ist_ihv"]:
            continue
        flaeche = seite["flaeche"]
        for m in ref_muster.finditer(flaeche):
            praefix = ("Abschnitt" if m.group(1).startswith("Abschnitt")
                       else "Anlage" if m.group(1).startswith("Anlage") else "Anhang")
            nummern = [m.group(2)] + re.findall(r"\d+(?:\.\d+)*", m.group(3) or "")
            for nr in nummern:
                n_refs += 1
                if nr not in vorhandene[praefix]:
                    befunde.append(Befund(
                        f"Verweis auf „{praefix} {nr}“, aber ein solcher Teil "
                        f"existiert im Dokument nicht.", seite["nummer"]))
    for nr, seite in sorted(set(dok.gleichungs_refs)):
        if nr not in dok.gleichungs_defs:
            befunde.append(Befund(
                f"Verweis auf Gleichung ({nr}), die nirgends definiert ist.", seite))
    return befunde, (f"{n_refs} Teil-Verweise und {len(set(dok.gleichungs_refs))} "
                     f"Gleichungs-Verweise geprüft.")


# ---------------------------------------------------------------- V-03
@check("V-03", "Inhaltsverzeichnis (Titel und Seitenzahlen)", "dokumentintern",
       "deterministisch")
def check_inhaltsverzeichnis(dok):
    """Das gedruckte IHV muss Titel und Seitenzahlen der tatsächlichen
    Überschriften wiedergeben. Der konstante Versatz zwischen gedruckter
    Seitenzahl und PDF-Seite (Deckblatt o. Ä.) wird automatisch bestimmt."""
    if not dok.ihv:
        raise CheckUebersprungen("Kein gedrucktes Inhaltsverzeichnis erkannt.")
    befunde = []

    def norm(s):
        return re.sub(r"\s+", " ", s).strip(" .").casefold()

    offsets = Counter()
    paare = []
    for e in dok.ihv:
        u = dok.ueberschrift(e.praefix, e.nummer)
        if u is None:
            befunde.append(Befund(
                f"IHV führt „{e.praefix} {e.nummer}: {e.titel}“, aber im Dokument "
                f"gibt es keine solche Überschrift.", e.pdf_seite))
            continue
        paare.append((e, u))
        offsets[u.seite - e.seite_lt_ihv] += 1
    versatz = offsets.most_common(1)[0][0] if offsets else 0
    for e, u in paare:
        if norm(e.titel) != norm(u.titel):
            befunde.append(Befund(
                f"Titel-Abweichung bei {e.praefix} {e.nummer}: IHV nennt "
                f"„{e.titel}“, die Überschrift lautet aber „{u.titel}“ "
                f"(Seite {u.seite}).", e.pdf_seite))
        if u.seite - e.seite_lt_ihv != versatz:
            befunde.append(Befund(
                f"Seitenzahl-Abweichung bei {e.praefix} {e.nummer}: IHV nennt "
                f"Seite {e.seite_lt_ihv}, die Überschrift steht aber auf "
                f"PDF-Seite {u.seite} (erwarteter Versatz {versatz:+d}).",
                e.pdf_seite))
    # Gegenrichtung: Überschriften, die im IHV fehlen
    ihv_menge = {(e.praefix, e.nummer) for e in dok.ihv}
    for u in dok.ueberschriften:
        if (u.praefix, u.nummer) not in ihv_menge:
            befunde.append(Befund(
                f"„{u.praefix} {u.nummer}: {u.titel}“ (Seite {u.seite}) fehlt im "
                f"Inhaltsverzeichnis.", u.seite, schwere=WARNUNG))
    return befunde, (f"{len(paare)} IHV-Einträge gegen Überschriften geprüft "
                     f"(Seitenversatz {versatz:+d}).")


# ---------------------------------------------------------------- V-04
def _symbol_stamm(token: str):
    """Reduziert ein Symbol-Vorkommen auf seinen Stamm: führende Buchstaben,
    ohne Geschlechts-Superskript und ohne Subskripte (x, t, e, m, Ziffern)."""
    m = re.match(r"^[^\W\d_]+", token, re.UNICODE)
    if not m:
        return None
    s = m.group(0)
    if len(s) > 1 and s[-1] in "MFW":
        s = s[:-1]
    while len(s) > 1 and s[-1] in "xtem":
        s = s[:-1]
    return s


@check("V-04", "Vollständigkeit des Symbolverzeichnisses", "dokumentintern",
       "deterministisch")
def check_symbolverzeichnis(dok):
    """Alle im Formelsatz wiederholt verwendeten Symbole müssen im
    Symbolverzeichnis erklärt sein."""
    if not dok.glossar_symbole:
        raise CheckUebersprungen("Kein Symbolverzeichnis/Glossar im Dokument erkannt.")
    erklaert = set()
    for g in dok.glossar_symbole:
        erklaert.add(g)
        stamm = _symbol_stamm(g)
        if stamm:
            erklaert.add(stamm)
    befunde = []
    gemeldet = set()
    for token, seiten in sorted(dok.symbol_vorkommen.items()):
        stamm = _symbol_stamm(token)
        if stamm is None or len(stamm) < 2 or token in erklaert or stamm in erklaert:
            continue
        # zusammengesetzte Symbole (z. B. a·x_gamma) gelten als erklärt,
        # wenn beide Bestandteile erklärt sind
        if any(token == s1 + s2 for s1 in erklaert for s2 in erklaert):
            continue
        if len(seiten) < 3 or len(set(seiten)) < 2 or stamm in gemeldet:
            continue
        gemeldet.add(stamm)
        schwere = FEHLER if len(seiten) >= 4 else WARNUNG
        befunde.append(Befund(
            f"Symbol „{token}“ (Stamm „{stamm}“) wird {len(seiten)}-mal verwendet "
            f"(u. a. Seiten {sorted(set(seiten))[:4]}), ist im Symbolverzeichnis "
            f"aber nicht erklärt.", sorted(set(seiten))[0], schwere=schwere))
    return befunde, (f"{len(dok.symbol_vorkommen)} verschiedene Symbole gegen "
                     f"{len(dok.glossar_symbole)} Glossar-Einträge geprüft.")


# ---------------------------------------------------------------- V-05
@check("V-05", "Tabellen-Identitäten und Rechnungszins", "dokumentintern",
       "deterministisch")
def check_tabellen_identitaeten(dok):
    """Prüft mathematische Identitäten direkt auf den Tabellen:
    (q+w)_x = q_x + w_x sowie den aus DQ_x implizierten Rechnungszins
    gegen die Angaben in Caption und Zins-Übersicht."""
    befunde = []
    geprueft = 0
    label_zu_geschlecht = {"Männer": "M", "Frauen": "F", "Kinder/Jugendliche": "K",
                           "Kinder": "K", "Jugendliche": "K"}
    deklariert = {}
    for label, prozent, seite, teil in dok.zins_deklarationen:
        deklariert[label_zu_geschlecht.get(label, label)] = (prozent, seite, teil)
    for t in dok.tabellen:
        idx_q, idx_w = t.spalten_indizes("q"), t.spalten_indizes("w")
        idx_qw, idx_dq = t.spalten_indizes("qw"), t.spalten_indizes("DQ")
        if idx_q and idx_w and idx_qw:
            for alter in sorted(t.zeilen):
                q, w, qw = (t.wert(alter, idx_q[0]), t.wert(alter, idx_w[0]),
                            t.wert(alter, idx_qw[0]))
                geprueft += 1
                if abs(q + w - qw) > 1.5e-6:
                    befunde.append(Befund(
                        f"{t.kurzname()}, Alter {alter}: (q+w)_x = "
                        f"{t.roh(alter, idx_qw[0])} stimmt nicht mit q_x + w_x = "
                        f"{fmt_de(q + w, 6)} überein.", t.seite_von(alter)))
        if idx_qw and idx_dq:
            vs = []
            for alter in sorted(t.zeilen):
                qw, dq = t.wert(alter, idx_qw[0]), t.wert(alter, idx_dq[0])
                if qw < 0.99 and dq > 0:
                    vs.append(dq / (1 - qw))
            if len(vs) >= 5:
                vs.sort()
                v = vs[len(vs) // 2]
                zins_impl = round((1 / v - 1) * 100, 2)
                if t.zins_prozent is not None and abs(zins_impl - t.zins_prozent) > 0.05:
                    befunde.append(Befund(
                        f"{t.kurzname()}: Caption nennt Rechnungszins "
                        f"{fmt_de(t.zins_prozent, 2)} %, aus den DQ_x-Werten folgt aber "
                        f"{fmt_de(zins_impl, 2)} %.", t.seiten[0]))
                if t.geschlecht in deklariert:
                    dekl, dekl_seite, dekl_teil = deklariert[t.geschlecht]
                    if abs(zins_impl - dekl) > 0.05:
                        befunde.append(Befund(
                            f"Zins-Übersicht ({dekl_teil}, Seite {dekl_seite}) nennt "
                            f"für „{t.geschlecht}“ {fmt_de(dekl, 2)} %, die Tabelle "
                            f"{t.kurzname()} ist aber erkennbar mit {zins_impl} % "
                            f"gerechnet (aus DQ_x = v·(1−(q+w)_x)).", dekl_seite))
    if geprueft == 0 and not befunde:
        raise CheckUebersprungen("Keine Tabelle mit q/w/(q+w)- oder DQ-Spalten erkannt.")
    return befunde, f"{geprueft} Zeilen-Identitäten geprüft."


# ---------------------------------------------------------------- V-06
# Ein Rückgang des Beitrags zwischen zwei Eintrittsaltern ist plausibel, wenn
# die TB ihn selbst für genau diesen Übergang begründet (typisch: Wegfall
# kostenbedingter Zuschläge ab Alter 65, § 8 Abs. 3 Satz 3 KVAV) und er klein
# bleibt. Der Wegfall prozentualer Zuschläge um wenige Prozentpunkte senkt den
# Beitrag um höchstens wenige Prozent (Beispiel-TB: Zuschläge 19,7 % -> 17,6 %,
# also höchstens rund 2,6 %); größere Rückgänge bleiben auffällig.
V06_MAX_BEGRUENDETER_RUECKGANG = 0.03

_RUECKGANG_BEGRUENDET_RE = re.compile(
    r"Eintrittsalter\s*(\d{1,3})\s*(?:zum|auf)\s*(?:Eintrittsalter\s*)?(\d{1,3})"
    r"[^.]{0,80}?\b(?:sink|fäll|fall|niedriger|geringer)", re.IGNORECASE)


def _begruendete_rueckgaenge(dok) -> dict[tuple[int, int], int]:
    """Übergänge (Alter a, Alter a+1), für die die TB einen Rückgang des
    Beitrags ausdrücklich begründet, mit der Seite der Begründung."""
    gefunden = {}
    for seite in dok.seiten:
        for m in _RUECKGANG_BEGRUENDET_RE.finditer(seite["flaeche"]):
            a1, a2 = int(m.group(1)), int(m.group(2))
            if a2 == a1 + 1:
                gefunden.setdefault((a1, a2), seite["nummer"])
    return gefunden


@check("V-06", "Monotonie der Beitragstabellen", "dokumentintern", "deterministisch")
def check_monotonie(dok):
    """Beiträge steigen in aller Regel mit dem Eintrittsalter. Ein Rückgang
    ist plausibel, wenn die TB ihn für genau diesen Übergang selbst begründet
    (z. B. Wegfall der unmittelbaren Abschlusskosten ab Alter 65) und er
    höchstens 3 % beträgt; jeder andere Rückgang ist prüfwürdig -> WARNUNG."""
    begruendet = _begruendete_rueckgaenge(dok)
    befunde = []
    plausibel = []          # (a1, a2, Rückgang in %, Seite der Begründung)
    geprueft = 0
    for t in dok.tabellen:
        for kanon in ("b", "P"):
            for idx in t.spalten_indizes(kanon):
                alter_liste = sorted(t.zeilen)
                for a1, a2 in zip(alter_liste, alter_liste[1:]):
                    if a2 != a1 + 1:
                        continue
                    geprueft += 1
                    w1, w2 = t.wert(a1, idx), t.wert(a2, idx)
                    # Rundungsbedingtes "Zittern" um wenige Cent ignorieren
                    if not (w1 > w2 and (w1 - w2) > max(0.05, 2e-4 * w1)):
                        continue
                    rueckgang = (w1 - w2) / w1
                    if (a1, a2) in begruendet and rueckgang <= V06_MAX_BEGRUENDETER_RUECKGANG:
                        plausibel.append((a1, a2, rueckgang, begruendet[(a1, a2)]))
                        continue
                    zusatz = (f" Die TB begründet zwar einen Rückgang bei {a1} → {a2} "
                              f"(Seite {begruendet[(a1, a2)]}), aber nicht in dieser Höhe "
                              f"({fmt_de(100 * rueckgang, 1)} %)."
                              if (a1, a2) in begruendet else
                              " Die TB begründet an keiner Stelle einen Rückgang bei "
                              "diesem Übergang.")
                    befunde.append(Befund(
                        f"{t.kurzname()}, Spalte {kanon}_x (Spalte {idx + 1}): "
                        f"Wert fällt von Alter {a1} ({t.roh(a1, idx)}) auf "
                        f"Alter {a2} ({t.roh(a2, idx)}).{zusatz} Bitte prüfen.",
                        t.seite_von(a1), schwere=WARNUNG))
    if geprueft == 0:
        raise CheckUebersprungen("Keine Beitrags-/Nettobeitragsspalten erkannt.")
    hinweis = f"{geprueft} Altersübergänge geprüft."
    if plausibel:
        uebergaenge = sorted({(a1, a2, s) for a1, a2, _, s in plausibel})
        groesster = max(r for _, _, r, _ in plausibel)
        anzahl = f"{len(plausibel)} Rückgang" if len(plausibel) == 1 else f"{len(plausibel)} Rückgänge"
        hinweis += (f" {anzahl} (höchstens "
                    f"{fmt_de(100 * groesster, 2)} %) beim in der TB begründeten Übergang "
                    + ", ".join(f"{a1} → {a2} (Begründung Seite {s})" for a1, a2, s in uebergaenge)
                    + " – plausibel, nicht gemeldet.")
    return befunde, hinweis


# ---------------------------------------------------------------- V-07
_ABGLEICHBARE_SPALTEN = {"q", "w", "qw", "DQ", "k", "K", "A", "a", "A12", "a12", "P", "b"}


@check("V-07", "Abgleich mit Referenzwerten (CSV)", "dokumentextern",
       "deterministisch")
def check_referenzwerte(dok):
    """Vergleicht alle erkannten Tabellenspalten mit den Referenz-CSVs.
    Eine Spalte gilt als zugeordnet, wenn >= 80 % der Werte (auf die im PDF
    gezeigten Nachkommastellen gerundet) exakt übereinstimmen – die
    verbleibenden Abweichungen sind dann echte Befunde."""
    if REFERENZ_CSV_ORDNER is None:
        raise CheckUebersprungen("Kein Referenzdaten-Ordner konfiguriert.")
    ordner = Path(REFERENZ_CSV_ORDNER)
    if not ordner.is_dir():
        raise CheckUebersprungen(f"Referenzdaten-Ordner nicht gefunden: {ordner}")
    referenzen = []  # (name, geschlecht, DataFrame mit kanonischen Spalten)
    for csv in sorted(ordner.glob("*.csv")):
        df = pd.read_csv(csv, sep=";")
        df.columns = [_CSV_KANON.get(c.strip(), c.strip()) for c in df.columns]
        if "x" not in df.columns:
            continue
        df = df.set_index(df["x"].astype(int))
        m = re.search(r"_([MFW])\b|_([MFW])\.csv$", csv.name, re.I)
        geschlecht = (m.group(1) or m.group(2)).upper() if m else None
        referenzen.append((csv.name, geschlecht, df))
    if not referenzen:
        raise CheckUebersprungen("Keine verwertbaren CSV-Dateien gefunden.")
    befunde = []
    zugeordnet, zellen = 0, 0
    for t in dok.tabellen:
        if not t.spalten or t.spalten[0] != "x":
            continue
        for idx, kanon in enumerate(t.spalten):
            if kanon not in _ABGLEICHBARE_SPALTEN:
                continue
            beste = None  # (quote, name, abweichungen, n)
            for name, _, df in referenzen:
                if kanon not in df.columns:
                    continue
                treffer, abweichungen, n = 0, [], 0
                for alter in sorted(t.zeilen):
                    if alter not in df.index:
                        continue
                    roh = t.roh(alter, idx)
                    toleranz = 0.5001 * 10 ** (-nachkommastellen(roh)) + 1e-9
                    csv_wert = float(df.loc[alter, kanon])
                    n += 1
                    if abs(de_zahl(roh) - csv_wert) <= toleranz:
                        treffer += 1
                    else:
                        abweichungen.append((alter, roh, csv_wert))
                if n >= 5:
                    quote = treffer / n
                    if beste is None or quote > beste[0]:
                        beste = (quote, name, abweichungen, n)
            if beste and beste[0] >= 0.8:
                zugeordnet += 1
                zellen += beste[3]
                for alter, roh, csv_wert in beste[2]:
                    befunde.append(Befund(
                        f"{t.kurzname()}, Alter {alter}, Spalte "
                        f"„{t.spalten[idx]}“ (Nr. {idx + 1}): PDF zeigt {roh}, "
                        f"Referenz ({beste[1]}) ergibt {fmt_de(csv_wert, nachkommastellen(roh))}.",
                        t.seite_von(alter)))
    if zugeordnet == 0:
        raise CheckUebersprungen(
            "Keine Tabellenspalte konnte einer Referenzspalte zugeordnet werden.")
    return befunde, (f"{zugeordnet} Spalten den Referenz-CSVs zugeordnet, "
                     f"{zellen} Zellen verglichen.")


# ---------------------------------------------------------------- V-08
@check("V-08", "Konsistenz benannter Parameter", "dokumentintern", "deterministisch")
def check_parameter_konsistenz(dok):
    """Sucht Zuweisungen wie 'G^M = 1901,0070 EUR' im Fließtext. Wird
    demselben Symbol an verschiedenen Stellen ein unterschiedlicher Wert
    zugewiesen, liegt ein Widerspruch im Dokument vor."""
    zuweisungen = defaultdict(list)  # (symbol, einheit) -> [(wert, roh, seite)]
    muster = re.compile(
        r"([GγσΩαi][A-Za-z0-9γ]{0,5})\s*=\s*"
        r"(\d{1,3}(?:\.\d{3})*,\d+|\d+,\d+)\s*(%|EUR)?(?!\s*[+\-·*/])")
    for seite in dok.seiten:
        if seite["ist_ihv"]:
            continue
        flaeche = seite["flaeche"]
        for m in muster.finditer(flaeche):
            symbol, roh, einheit = m.group(1), m.group(2), m.group(3) or ""
            zuweisungen[(symbol, einheit)].append((de_zahl(roh), roh, seite["nummer"]))
    befunde = []
    n = 0
    for (symbol, einheit), werte in sorted(zuweisungen.items()):
        n += len(werte)
        verschieden = sorted({w for w, _, _ in werte})
        if len(verschieden) > 1:
            details = "; ".join(f"{roh} {einheit} (Seite {s})" for _, roh, s in werte)
            befunde.append(Befund(
                f"Widersprüchliche Werte für „{symbol}“: {details}.",
                werte[0][2]))
    return befunde, f"{n} Parameter-Zuweisungen zu {len(zuweisungen)} Symbolen geprüft."


# ---------------------------------------------------------------- V-09
# Wissensbasis: fachlicher Kontext -> einschlägige Norm (nur gesicherte Zuordnungen)
_NORM_WISSEN = [
    (r"Höchstrechnungszins|Rechnungszins", "KVAV", "4",
     "Der Rechnungszins in der PKV ist in § 4 KVAV geregelt."),
    (r"Sicherheitszuschlag", "KVAV", "7",
     "Der Sicherheitszuschlag (mind. 5 %) ist in § 7 KVAV geregelt."),
    (r"[Gg]esetzlichen? Beitragszuschlag", "VAG", "149",
     "Der gesetzliche Beitragszuschlag (10 %) ist in § 149 VAG geregelt."),
    (r"Direktgutschrift", "VAG", "150",
     "Die Direktgutschrift ist in § 150 VAG geregelt."),
]


@check("V-09", "Gesetzes- und Paragrafen-Prüfung", "dokumentextern",
       "deterministisch")
def check_gesetze(dok):
    """(a) Normzitate unmittelbar hinter bekannten Fachbegriffen müssen zur
    Wissensbasis passen. (b) Der Sicherheitszuschlag muss den gesetzlichen
    Mindestsatz von 5 % (§ 7 KVAV) erreichen und einheitlich angegeben sein."""
    befunde = []
    n = 0
    for seite in dok.seiten:
        if seite["ist_ihv"]:
            continue
        flaeche = seite["flaeche"]
        for schluessel, norm_soll, nr_soll, erklaerung in _NORM_WISSEN:
            for m in re.finditer(
                    rf"({schluessel})[^§.]{{0,60}}§\s*(\d+[a-z]?)"
                    rf"(?:\s*Abs\.\s*\d+)?(?:\s*(?:Satz|Nr\.)\s*\d+)?\s+"
                    rf"([A-ZÄÖÜ][A-Za-zÄÖÜäöü]*)", flaeche):
                n += 1
                nr_ist, norm_ist = m.group(2), m.group(3)
                if norm_ist != norm_soll or nr_ist != nr_soll:
                    befunde.append(Befund(
                        f"„{m.group(1)}“ wird mit „§ {nr_ist} {norm_ist}“ belegt – "
                        f"einschlägig ist jedoch § {nr_soll} {norm_soll}. "
                        f"({erklaerung})", seite["nummer"]))
    # Numerische Prüfung: Sicherheitszuschlag >= 5 % und einheitlich
    sigma_werte = defaultdict(list)
    for seite in dok.seiten:
        if seite["ist_ihv"]:
            continue
        flaeche = seite["flaeche"]
        for m in re.finditer(r"Sicherheitszuschlag[^%]{0,80}?(\d+,\d+)\s*%", flaeche):
            sigma_werte[de_zahl(m.group(1))].append(seite["nummer"])
    if sigma_werte:
        n += sum(len(v) for v in sigma_werte.values())
        for wert, seiten in sorted(sigma_werte.items()):
            if wert < 5.0:
                befunde.append(Befund(
                    f"Sicherheitszuschlag von {fmt_de(wert, 1)} % (Seiten {sorted(set(seiten))}) "
                    f"unterschreitet den gesetzlichen Mindestsatz von 5 % der "
                    f"Bruttoprämie (§ 7 KVAV).", seiten[0]))
        if len(sigma_werte) > 1:
            details = "; ".join(f"{fmt_de(w, 1)} % (Seiten {sorted(set(s))})"
                                for w, s in sorted(sigma_werte.items()))
            befunde.append(Befund(
                f"Der Sicherheitszuschlag wird uneinheitlich angegeben: {details}.",
                min(min(s) for s in sigma_werte.values())))
    return befunde, f"{n} Normbezüge bzw. Zuschlagsangaben geprüft."


# ---------------------------------------------------------------- V-10
@check("V-10", "Referenzen auf Rahmen-Berechnungsgrundlagen", "dokumentextern",
       "deterministisch")
def check_rahmen_tb(dok):
    """Jede referenzierte Rahmen-TB muss im Rahmen-TB-Register des Dokuments
    mit Versionsangabe geführt sein; abweichende Versionsangaben sind Fehler."""
    if not dok.rahmen_tb_register and not dok.rahmen_tb_referenzen:
        raise CheckUebersprungen("Keine Rahmen-TB-Referenzen im Dokument gefunden.")
    befunde = []
    for name, (version, seite) in sorted(dok.rahmen_tb_register.items()):
        if not version:
            befunde.append(Befund(
                f"Rahmen-TB „{name}“ ist im Register ohne Versionsangabe geführt.",
                seite, schwere=WARNUNG))
    for name, version, seite in dok.rahmen_tb_referenzen:
        if name not in dok.rahmen_tb_register:
            befunde.append(Befund(
                f"Referenz auf „{name}“ (Seite {seite}), die im "
                f"Rahmen-TB-Register nicht geführt wird.", seite))
        elif version and dok.rahmen_tb_register[name][0] \
                and version != dok.rahmen_tb_register[name][0]:
            befunde.append(Befund(
                f"Referenz auf „{name}“ in Version {version} (Seite {seite}), "
                f"das Register nennt aber Version {dok.rahmen_tb_register[name][0]}.",
                seite))
    return befunde, (f"{len(dok.rahmen_tb_referenzen)} Referenzen gegen ein Register "
                     f"mit {len(dok.rahmen_tb_register)} Einträgen geprüft.")




# ---------------------------------------------------------------- V-11
@check("V-11", "Versions- und Stichtagskonsistenz", "dokumentintern",
       "deterministisch")
def check_versionskonsistenz(dok):
    """Änderungsnummer und Gültigkeitsdatum müssen im ganzen Dokument
    übereinstimmen (Kopfzeilen, Titelseite, Änderungshistorie); die
    Änderungshistorie muss den aktuellen Stand enthalten."""
    nummern, daten = defaultdict(set), defaultdict(set)
    for seite in dok.seiten:
        fl = seite["flaeche"]
        for m in re.finditer(r"Änderungs(?:nr\.|nummer)\s*:?\s*(\d+)", fl):
            nummern[int(m.group(1))].add(seite["nummer"])
        for m in re.finditer(
                r"Änderungs(?:nr\.|nummer)\s*:?\s*\d+\s*\(?\s*ab\s+(\d{2}\.\d{2}\.\d{4})", fl):
            daten[m.group(1)].add(seite["nummer"])
    if not nummern:
        raise CheckUebersprungen("Keine Änderungsnummern-Angaben im Dokument gefunden.")
    befunde = []
    # Der dokumentweite Stand ist die mit Abstand häufigste Angabe (Kopfzeilen).
    # Vereinzelte andere Nummern sind meist historische Bezüge ("seit Änderung
    # Nr. 11") – erst eine ANDERE Nummer auf vielen Seiten ist eine echte
    # Inkonsistenz.
    aktuelle = max(nummern, key=lambda n: len(nummern[n]))
    schwelle = max(3, 0.2 * len(nummern[aktuelle]))
    for n, seiten in sorted(nummern.items()):
        if n != aktuelle and len(seiten) >= schwelle:
            befunde.append(Befund(
                f"Uneinheitlicher Dokumentstand: Änderungsnr. {n} auf "
                f"{len(seiten)} Seiten (z. B. Seite {min(seiten)}), überwiegend "
                f"gilt aber Nr. {aktuelle}.", min(seiten)))
    datum_haeufigste = max(daten, key=lambda d: len(daten[d])) if daten else None
    for d, seiten in sorted(daten.items()):
        if d != datum_haeufigste and len(seiten) >= schwelle:
            befunde.append(Befund(
                f"Uneinheitliches Gültigkeitsdatum: {d} auf {len(seiten)} Seiten "
                f"(z. B. Seite {min(seiten)}), überwiegend gilt {datum_haeufigste}.",
                min(seiten)))
    # Änderungshistorie: höchste dort gelistete Nummer = aktueller Stand?
    historie_max = None
    for u in dok.ueberschriften:
        if "änderungshistorie" not in u.titel.casefold():
            continue
        for seite in dok.seiten:
            if seite["nummer"] != u.seite:
                continue
            # Historienzeilen haben die Form "<Nr.> <Datum> <Beschreibung>"
            gefunden = re.findall(r"\b(\d{1,3})\s+\d{2}\.\d{2}\.\d{4}", seite["flaeche"])
            if gefunden:
                historie_max = max(int(g) for g in gefunden)
    if historie_max is not None and historie_max != aktuelle:
        befunde.append(Befund(
            f"Die Änderungshistorie endet bei Nr. {historie_max}, das Dokument "
            f"trägt aber Änderungsnr. {aktuelle}.", None))
    return befunde, (f"{sum(len(s) for s in nummern.values())} Nummern- und "
                     f"{sum(len(s) for s in daten.values())} Datumsangaben geprüft "
                     f"(Dokumentstand: Nr. {aktuelle})"
                     + (f"; Historie bis Nr. {historie_max}." if historie_max else "."))


# ---------------------------------------------------------------- V-12
_WERTEBEREICHE = {"q": (0.0, 1.0), "w": (0.0, 1.0), "qw": (0.0, 1.0), "DQ": (0.0, 1.0)}
_NICHT_NEGATIV = {"k", "K", "A", "a", "A12", "a12", "P", "b", "l"}


@check("V-12", "Vollständigkeit und Wertebereiche der Alterstabellen",
       "dokumentintern", "deterministisch")
def check_tafel_vollstaendigkeit(dok):
    """Alterstabellen dürfen keine Alterslücken haben, müssen das im Text
    deklarierte Endalter erreichen und nur plausible Werte enthalten
    (Wahrscheinlichkeiten in [0,1], Diskontfaktoren in (0,1), keine
    negativen Barwerte/Beiträge)."""
    tabellen = [t for t in dok.tabellen
                if t.spalten and t.spalten[0] == "x" and len(t.zeilen) >= 10]
    if not tabellen:
        raise CheckUebersprungen("Keine Alterstabellen erkannt.")
    befunde = []
    n_werte = 0
    for t in tabellen:
        alter = sorted(t.zeilen)
        luecken = [a for a in range(alter[0], alter[-1]) if a not in t.zeilen]
        if luecken:
            befunde.append(Befund(
                f"{t.kurzname()}: Alterslücke(n) bei {luecken[:6]}"
                f"{' ...' if len(luecken) > 6 else ''} (Tafel läuft von "
                f"{alter[0]} bis {alter[-1]}).", t.seiten[0]))
        # deklariertes Endalter im Umfeld der Tabelle
        for seite in dok.seiten:
            if seite["nummer"] == t.seiten[0]:
                m = re.search(r"Endalter[^0-9]{0,40}(\d{2,3})", seite["flaeche"])
                if m and int(m.group(1)) != alter[-1]:
                    befunde.append(Befund(
                        f"{t.kurzname()}: deklariertes Endalter {m.group(1)}, "
                        f"die Tafel endet aber bei Alter {alter[-1]}.", t.seiten[0]))
        for idx, kanon in enumerate(t.spalten[1:], start=1):
            grenzen = _WERTEBEREICHE.get(kanon)
            for a in alter:
                w = t.wert(a, idx)
                n_werte += 1
                if grenzen and not (grenzen[0] <= w <= grenzen[1]):
                    befunde.append(Befund(
                        f"{t.kurzname()}, Alter {a}: Wert {t.roh(a, idx)} der "
                        f"Spalte „{kanon}“ liegt außerhalb von [0, 1].",
                        t.seite_von(a)))
                elif kanon in _NICHT_NEGATIV and w < 0:
                    befunde.append(Befund(
                        f"{t.kurzname()}, Alter {a}: negativer Wert "
                        f"{t.roh(a, idx)} in Spalte „{kanon}“.", t.seite_von(a)))
    return befunde, (f"{len(tabellen)} Alterstabellen mit {n_werte} Werten auf "
                     f"Lücken, Endalter und Wertebereiche geprüft.")


# ---------------------------------------------------------------- V-13
# Zuordnung der Datenbankspalten zu den kanonischen Namen der PDF-Tabellen
_DB_KANON = {"x": "x", "qx": "q", "q_x": "q", "wx": "w", "w_x": "w",
             "kopfschaden": "K", "k_x": "K", "kx": "K"}


@check("V-13", "Abgleich mit der Referenzdatenbank (SQLite)", "dokumentextern",
       "deterministisch")
def check_datenbank(dok):
    """Vergleicht die in der TB abgedruckten Sterbe- und
    Stornowahrscheinlichkeiten sowie Kopfschäden zellgenau mit der
    SQLite-Referenzdatenbank; die Summenspalte (q+w)_x wird gegen die aus
    der Datenbank abgeleitete Summe q_x + w_x geprüft."""
    if REFERENZ_DB_PFAD is None:
        raise CheckUebersprungen("Keine Referenzdatenbank konfiguriert.")
    db_pfad = Path(REFERENZ_DB_PFAD)
    if not db_pfad.exists():
        raise CheckUebersprungen(f"Referenzdatenbank nicht gefunden: {db_pfad}")
    verbindung = sqlite3.connect(str(db_pfad))
    # Referenz einlesen: {geschlecht: {alter: {kanon: wert}}}
    referenz = defaultdict(dict)
    for (tabelle,) in verbindung.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"):
        spalten = [z[1] for z in verbindung.execute(f'PRAGMA table_info("{tabelle}")')]
        kanon = {s: _DB_KANON.get(s.lower()) for s in spalten}
        if "geschlecht" not in [s.lower() for s in spalten] or                 "x" not in kanon.values():
            continue
        for zeile in verbindung.execute(f'SELECT * FROM "{tabelle}"'):
            werte = dict(zip(spalten, zeile))
            geschlecht = str(werte.get("geschlecht", "")).upper()
            alter = next(int(w) for s, w in werte.items() if kanon.get(s) == "x")
            eintrag = {kanon[s]: float(w) for s, w in werte.items()
                       if kanon.get(s) in ("q", "w", "K")}
            eintrag["qw"] = eintrag.get("q", 0) + eintrag.get("w", 0)
            referenz[geschlecht][alter] = eintrag
    verbindung.close()
    if not referenz:
        raise CheckUebersprungen(
            "Keine verwertbare Tabelle (geschlecht/x/qx/wx/...) in der Datenbank.")
    befunde = []
    zugeordnet, zellen = 0, 0
    for t in dok.tabellen:
        if not t.spalten or t.spalten[0] != "x":
            continue
        for idx, kanon in enumerate(t.spalten):
            if kanon not in ("q", "w", "K", "qw"):
                continue
            beste = None  # (quote, geschlecht, abweichungen, n)
            for geschlecht, eintraege in referenz.items():
                treffer, abweichungen, n = 0, [], 0
                for alter in sorted(t.zeilen):
                    if alter not in eintraege or kanon not in eintraege[alter]:
                        continue
                    roh = t.roh(alter, idx)
                    toleranz = 0.5001 * 10 ** (-nachkommastellen(roh)) + 1e-9
                    n += 1
                    if abs(de_zahl(roh) - eintraege[alter][kanon]) <= toleranz:
                        treffer += 1
                    else:
                        abweichungen.append((alter, roh, eintraege[alter][kanon]))
                if n >= 5:
                    quote = treffer / n
                    if beste is None or quote > beste[0]:
                        beste = (quote, geschlecht, abweichungen, n)
            if beste and beste[0] >= 0.8:
                zugeordnet += 1
                zellen += beste[3]
                for alter, roh, soll in beste[2]:
                    zusatz = (" (Datenbank: q_x + w_x)" if kanon == "qw" else "")
                    befunde.append(Befund(
                        f"{t.kurzname()}, Alter {alter}, Spalte „{kanon}“ "
                        f"(Nr. {idx + 1}): PDF zeigt {roh}, die Referenzdatenbank "
                        f"({beste[1]}) ergibt {fmt_de(soll, nachkommastellen(roh))}"
                        f"{zusatz}.", t.seite_von(alter)))
    if zugeordnet == 0:
        raise CheckUebersprungen(
            "Keine Tabellenspalte konnte der Datenbank zugeordnet werden.")
    return befunde, (f"{zugeordnet} Spalten der Datenbank zugeordnet, "
                     f"{zellen} Zellen verglichen ({db_pfad.name}).")

print(f"{len(CHECKS)} deterministische Checks registriert.")

13 deterministische Checks registriert.


## 6 | LLM-Anbindung (Claude)

Die KI-Checks senden Textauszüge an das konfigurierte Claude-Modell und
erzwingen über **Structured Outputs** (`output_config.format` mit
JSON-Schema) maschinenlesbare Antworten – es ist kein fehleranfälliges
Parsen von Freitext nötig.

- Der API-Key kommt **ausschließlich** aus der Umgebungsvariable
  `ANTHROPIC_API_KEY`; ohne Key werden die LLM-Checks übersprungen.
- Ein **Antwort-Cache** auf Platte macht wiederholte Notebook-Läufe
  kostenlos und reproduzierbar; ein Aufruf-Limit begrenzt die Kosten.
- **Werkzeuge (Tool Use)**: Ein Check kann Claude Werkzeuge mitgeben; Claude
  entscheidet selbst, welche Aufrufe es braucht (L-04: Gesetzestexte
  nachschlagen). Das Notebook führt die Aufrufe aus und reicht die Ergebnisse
  zurück, bis die JSON-Antwort vorliegt.
- Modellwahl in der Konfigurationszelle: `claude-haiku-4-5` (erster Test,
  schnell/günstig) oder `claude-sonnet-5` (höhere Erkennungsqualität).

In [6]:
# =============================================================================
# LLM-Infrastruktur: Claude-Anbindung mit strukturiertem JSON-Output
# =============================================================================
# Der API-Key wird AUSSCHLIESSLICH aus der Umgebungsvariable ANTHROPIC_API_KEY
# gelesen und niemals im Notebook gespeichert. Fehlt der Key, melden sich die
# LLM-Checks als ÜBERSPRUNGEN – alle übrigen Checks laufen normal weiter.
#
# Werkzeuge (Tool Use): Ein Check kann Claude Werkzeuge mitgeben, z. B. das
# Nachschlagen eines Gesetzestexts (L-04). Claude entscheidet selbst, welche
# Aufrufe es braucht; das Notebook führt sie aus, schickt die Ergebnisse
# zurück und wiederholt das, bis die JSON-Antwort vorliegt.

from concurrent.futures import ThreadPoolExecutor
from typing import Callable

import anthropic


def llm_verfuegbar() -> bool:
    return bool(os.environ.get("ANTHROPIC_API_KEY"))


_LLM_SYSTEM = (
    "Du bist ein erfahrener Aktuar der privaten Krankenversicherung (PKV) und "
    "prüfst Technische Berechnungsgrundlagen (TB). Du arbeitest präzise und "
    "konservativ: Du meldest nur Befunde, bei denen du dir sicher bist. "
    "Deutsche Zahlen (z. B. 1.234,56) wandelst du in JSON in Zahlen mit "
    "Dezimalpunkt um. Du antwortest ausschließlich im geforderten JSON-Format."
)

_llm_client = anthropic.Anthropic() if llm_verfuegbar() else None
_llm_aufrufe = 0
MAX_WERKZEUG_RUNDEN = 5   # danach muss Claude ohne weitere Werkzeugaufrufe antworten

# Antwort-Cache auf Platte: identische Anfragen (Modell + Prompt + Schema)
# werden bei erneuten Notebook-Läufen nicht erneut berechnet/berechnet.
try:
    _llm_cache = json.loads(Path(LLM_CACHE_PFAD).read_text(encoding="utf-8"))
except (FileNotFoundError, json.JSONDecodeError):
    _llm_cache = {}


@dataclass(frozen=True)
class Werkzeug:
    """Ein Werkzeug, das Claude während einer Anfrage aufrufen darf: Name,
    Beschreibung, JSON-Schema der Eingabe und die Python-Funktion, die einen
    Aufruf ausführt (Rückgabe: Text für Claude; eine Ausnahme wird Claude als
    Fehlermeldung zurückgegeben)."""
    name: str
    beschreibung: str
    eingabe_schema: dict = field(hash=False, compare=False)
    ausfuehren: Callable[..., str] = field(hash=False, compare=False)

    def definition(self) -> dict:
        return {"name": self.name, "description": self.beschreibung,
                "input_schema": self.eingabe_schema, "strict": True}


def werkzeug_ergebnisse(werkzeuge, inhalt) -> list[dict]:
    """Führt alle Werkzeugaufrufe einer Antwort gleichzeitig aus und liefert
    die tool_result-Blöcke – zusammen für EINE user-Nachricht."""
    nach_name = {w.name: w for w in werkzeuge}
    aufrufe = [b for b in inhalt if getattr(b, "type", None) == "tool_use"]

    def ausfuehren(aufruf):
        try:
            if aufruf.name not in nach_name:
                raise ValueError(f"Unbekanntes Werkzeug „{aufruf.name}“.")
            ergebnis = nach_name[aufruf.name].ausfuehren(**aufruf.input)
            return {"type": "tool_result", "tool_use_id": aufruf.id, "content": str(ergebnis)}
        except Exception as fehler:
            return {"type": "tool_result", "tool_use_id": aufruf.id,
                    "content": f"Fehler: {fehler}", "is_error": True}

    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(ausfuehren, aufrufe))


def llm_json(auftrag: str, text: str, schema: dict, max_tokens: int = 8000,
             bild_b64: str | None = None, werkzeuge: list | None = None) -> dict:
    """Stellt eine Anfrage an das konfigurierte Claude-Modell und erzwingt
    über 'output_config.format' (Structured Outputs) eine Antwort, die dem
    übergebenen JSON-Schema entspricht. Optional kann ein Seitenbild (PNG,
    Base64) mitgegeben werden – Grundlage der Vision-Checks – sowie eine
    Liste von Werkzeugen, die Claude unterwegs aufrufen darf."""
    global _llm_aufrufe
    if _llm_client is None:
        raise CheckUebersprungen(
            "Kein ANTHROPIC_API_KEY in der Umgebung – LLM-Check übersprungen.")
    bild_hash = hashlib.sha256(bild_b64.encode("ascii")).hexdigest() if bild_b64 else None
    teile = [MODELL, auftrag, text, schema, bild_hash]
    if werkzeuge:   # Anfragen ohne Werkzeuge behalten ihren bisherigen Schlüssel
        teile.append(sorted(w.name for w in werkzeuge))
    schluessel = hashlib.sha256(
        json.dumps(teile, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    if schluessel in _llm_cache:
        return _llm_cache[schluessel]
    if _llm_aufrufe >= MAX_LLM_AUFRUFE:
        raise CheckUebersprungen(
            f"Sicherheitslimit von {MAX_LLM_AUFRUFE} LLM-Aufrufen erreicht.")
    _llm_aufrufe += 1
    inhalt = [{"type": "text", "text": f"{auftrag}\n\n---\n\n{text}"}]
    if bild_b64:
        inhalt.insert(0, {"type": "image", "source": {
            "type": "base64", "media_type": "image/png", "data": bild_b64}})
    nachrichten = [{"role": "user", "content": inhalt}]
    for runde in range(MAX_WERKZEUG_RUNDEN + 1):
        zusatz = {}
        if werkzeuge:
            zusatz["tools"] = [w.definition() for w in werkzeuge]
            if runde == MAX_WERKZEUG_RUNDEN:
                zusatz["tool_choice"] = {"type": "none"}   # jetzt antworten
        antwort = _llm_client.messages.create(
            model=MODELL,
            max_tokens=max_tokens,
            system=_LLM_SYSTEM,
            messages=nachrichten,
            output_config={"format": {"type": "json_schema", "schema": schema}},
            **zusatz,
        )
        if antwort.stop_reason != "tool_use":
            break
        nachrichten.append({"role": "assistant", "content": antwort.content})
        nachrichten.append({"role": "user",
                            "content": werkzeug_ergebnisse(werkzeuge, antwort.content)})
    if antwort.stop_reason == "max_tokens":
        raise RuntimeError("LLM-Antwort wurde abgeschnitten (max_tokens erreicht).")
    antwort_text = next(b.text for b in antwort.content if b.type == "text")
    ergebnis = json.loads(antwort_text)
    _llm_cache[schluessel] = ergebnis
    Path(LLM_CACHE_PFAD).write_text(
        json.dumps(_llm_cache, ensure_ascii=False), encoding="utf-8")
    return ergebnis


def _schema_liste(feld: str, eigenschaften: dict, pflicht: list) -> dict:
    """Hilfsfunktion: Schema für {'<feld>': [ {...}, ... ]} (Structured Outputs
    verlangt 'additionalProperties': false und vollständige 'required'-Listen)."""
    return {
        "type": "object",
        "properties": {feld: {"type": "array", "items": {
            "type": "object", "properties": eigenschaften,
            "required": pflicht, "additionalProperties": False}}},
        "required": [feld], "additionalProperties": False,
    }


if llm_verfuegbar():
    print(f"LLM-Checks aktiv: Modell {MODELL} (Cache: {LLM_CACHE_PFAD}, "
          f"max. {MAX_LLM_AUFRUFE} Aufrufe)")
else:
    print("Hinweis: ANTHROPIC_API_KEY ist nicht gesetzt – die LLM-gestützten "
          "Checks werden übersprungen, alle deterministischen Checks laufen.")

LLM-Checks aktiv: Modell claude-sonnet-5 (Cache: llm_cache.json, max. 120 Aufrufe)


## 7 | LLM-gestützte Checks

| Check | prüft |
|---|---|
| L-01 | Rechtschreibung des Fließtexts (konservativ; PDF-Extraktionsartefakte ausgenommen) |
| L-02 | **Formel-Extraktion & Verifikation**: Claude extrahiert Formeln, eingesetzte Zahlen und behauptete Ergebnisse; das Notebook rechnet jede Formel in Python nach |
| L-03 | Konsistenz zwischen Abschnitten und gleichnummerigen Anlagen sowie den Beispielrechnungen (Werte, Parameter, Vorzeichen von Formeln) |
| L-04 | **Normzitate am Gesetzestext**: Claude schlägt per Werkzeugaufruf (Tool Use) den amtlichen Wortlaut der zitierten – und ggf. der eigentlich einschlägigen – Paragrafen auf gesetze-im-internet.de nach und prüft Paragraf, Absatz und Nummer; ohne Internet dient die gespeicherte Kopie `normtexte.json` |
| L-05 | **Vision-Prüfung des Formelsatzes**: formelhaltige Seiten werden als Bild geprüft – Widersprüche zwischen Formel und Beschreibung sowie Vorzeichen-/Strukturfehler, die im zerlegten PDF-Text-Layer unsichtbar sind |
| L-06 | **Grafik-Tabellen-Abgleich**: Claude liest die Kurven **blind** vom Seitenbild ab (ohne die Tabellenwerte zu kennen – gegen den Verankerungseffekt); der Vergleich mit den Tabellenspalten erfolgt anschließend deterministisch in Python, gemeldet werden nur krasse Abweichungen (Median > 20 %), die Ablesegenauigkeit wird transparent ausgewiesen |

**Modellwahl in der Praxis** (Validierung gegen die 13 dokumentierten
Fehler der Testvariante, Juli 2026): **Sonnet 5 fand alle 13 Fehler** –
ein gedrehtes Vorzeichen im Formelsatz, das sich in keinem Zahlenwert
niederschlägt, ausschließlich über die Vision-Prüfung L-05, und die um
Faktor 1,5 überhöhte Frauen-Kurve der Profil-Grafik über den
Grafik-Tabellen-Abgleich L-06 – und meldete auf der fehlerfreien Fassung
keine Fehler. Deren Warnungen betreffen vor allem Normzitate, deren
Absatz laut amtlichem Wortlaut nicht genau passt (L-04, Stand November 2026). Haiku 4.5 fand auf dem früheren Stand (12 Fehler)
elf davon (die Vision-Prüfung erkennt das Vorzeichen dort nicht) und
produziert in den Konsistenz-Checks einige Schein-Warnungen. Empfehlung:
Haiku 4.5 für schnelle, günstige Zwischenprüfungen; Sonnet 5 für den
maßgeblichen Prüfbericht.

In [7]:
# =============================================================================
# LLM-gestützte Checks (Claude, strukturierter JSON-Output)
# =============================================================================

def _prosa_zeilen(dok):
    """Liefert je Seite die 'Prosa'-Zeilen: wiederkehrende Kopfzeilen und
    zahlenlastige Tabellenzeilen werden entfernt, Silbentrennung aufgehoben."""
    haeufigkeit = Counter()
    for seite in dok.seiten:
        for z in set(seite["zeilen"]):
            haeufigkeit[z.strip()] += 1
    wiederkehrend = {z for z, n in haeufigkeit.items() if n >= dok.seitenzahl * 0.5}
    ergebnis = []
    for seite in dok.seiten:
        if seite["ist_ihv"]:
            continue
        zeilen = []
        for z in seite["zeilen"]:
            s = z.strip()
            if not s or s in wiederkehrend:
                continue
            buchstaben = sum(c.isalpha() for c in s)
            if buchstaben < max(6, 0.5 * len(s)):
                # kurze Fortsetzungszeile nach Silbentrennung trotzdem behalten,
                # sonst entstehen künstliche Wortbrüche in der Textfläche
                if not (zeilen and zeilen[-1].rstrip().endswith("-") and buchstaben >= 2):
                    continue
            zeilen.append(z)
        if zeilen:
            ergebnis.append((seite["nummer"], zeilen_zu_flaeche("\n".join(zeilen))))
    return ergebnis


def _text_chunks(seiten_texte, max_zeichen=7000):
    """Fasst (Seite, Text)-Paare zu Chunks mit [Seite N]-Markern zusammen."""
    chunks, aktuell = [], ""
    for nummer, text in seiten_texte:
        stueck = f"\n[Seite {nummer}]\n{text}"
        if aktuell and len(aktuell) + len(stueck) > max_zeichen:
            chunks.append(aktuell)
            aktuell = ""
        aktuell += stueck
    if aktuell:
        chunks.append(aktuell)
    return chunks


# ---------------------------------------------------------------- L-01
def _ist_extraktionsartefakt(falsch: str, richtig: str) -> bool:
    """Verwirft LLM-Meldungen, die keine echten Rechtschreibfehler sind:
    PDF-Extraktionsartefakte (zusammengeklebte Wörter, Formelsymbole,
    Umbruchreste), reine Interpunktionsvorschläge und Wortwahl-Vorschläge
    (völlig anderes Wort statt kleiner Buchstabenkorrektur)."""
    import difflib
    f, r = falsch.strip(), richtig.strip()
    if re.search(r"[^A-Za-zÄÖÜäöüß\-]", f):
        return True   # enthält Formel-/Sonderzeichen -> Symbol, kein Wort
    norm = lambda s: re.sub(r"[\s/,;.:_^]+", "", s).casefold()
    if norm(f) == norm(r):
        return True   # nur Leerzeichen/Interpunktion/Notation geändert -> Artefakt
    if re.search(r"-[A-ZÄÖÜ]", f):
        return True   # Zeilenumbruch-/Trennungsrest im Wort
    if len(f) > len(r) and f.endswith(r) and len(f) - len(r) <= 2:
        return True   # angeklebtes Formelsymbol (z. B. 'ije' -> 'je')
    if difflib.SequenceMatcher(None, f.casefold(), r.casefold()).ratio() < 0.8:
        return True   # anderes Wort vorgeschlagen -> Wortwahl, kein Tippfehler
    return False


@check("L-01", "Rechtschreibprüfung", "dokumentintern", "LLM")
def check_rechtschreibung(dok):
    """Claude prüft den Fließtext seitenweise auf eindeutige
    Rechtschreibfehler; PDF-Extraktionsartefakte werden ausgenommen."""
    schema = _schema_liste("fehler", {
        "falsch": {"type": "string"}, "richtig": {"type": "string"},
        "seite": {"type": "integer"}, "kontext": {"type": "string"}},
        ["falsch", "richtig", "seite", "kontext"])
    auftrag = (
        "Prüfe den folgenden Auszug aus einer TB (PDF-Textextraktion) auf "
        "eindeutige deutsche RECHTSCHREIBFEHLER (fehlende, vertauschte oder "
        "überzählige Buchstaben).\n"
        "KEINE Fehler sind: PKV-Fachbegriffe und zusammengesetzte Fachwörter, "
        "Abkürzungen, Symbole, Eigennamen – und vor allem Extraktionsartefakte: "
        "Formelsymbole kleben oft am Folgewort (z. B. 'iwird', 'xjährig', "
        "'vund') oder Leerzeichen fehlen an Formel- und Silbengrenzen. Melde "
        "ein Wort nur, wenn es auch unabhängig von solchen Artefakten falsch "
        "geschrieben ist. Keine Grammatik-, Numerus- oder Stilhinweise "
        "(z. B. Singular/Plural, Getrennt-/Zusammenschreibung) – nur Wörter, "
        "die in keiner Form korrekt geschrieben sind. Gib je Fund die Seite "
        "aus dem letzten davor stehenden [Seite N]-Marker an.")
    befunde, geprüfte_chunks = [], 0
    gesehen = set()
    for chunk in _text_chunks(_prosa_zeilen(dok), max_zeichen=4500):
        antwort = llm_json(auftrag, chunk, schema)
        geprüfte_chunks += 1
        for f in antwort.get("fehler", []):
            kennung = f["falsch"].casefold()
            if kennung in gesehen or _ist_extraktionsartefakt(f["falsch"], f["richtig"]):
                continue
            # Grounding: Das gemeldete Wort muss wörtlich im Umfeld der
            # angegebenen Seite stehen – sonst hat das Modell es falsch
            # abgeschrieben (halluziniert) und der Befund ist wertlos.
            seite = f.get("seite")
            umfeld = [s["flaeche"] for s in dok.seiten
                      if seite and abs(s["nummer"] - seite) <= 1]
            if not any(f["falsch"] in t for t in (umfeld or
                       [s["flaeche"] for s in dok.seiten])):
                continue
            gesehen.add(kennung)
            befunde.append(Befund(
                f"Rechtschreibfehler „{f['falsch']}“ (richtig: „{f['richtig']}“) "
                f"im Kontext: „...{f['kontext'][:90]}...“", f.get("seite")))
    return befunde, f"{geprüfte_chunks} Text-Chunks mit {MODELL} geprüft."


# ---------------------------------------------------------------- L-02
def _zahl_steht_im_dokument(dok, wert: float, seite: int | None, umfeld: int = 1) -> bool:
    """Grounding-Anker gegen LLM-Halluzinationen: Eine extrahierte Zahl wird
    nur akzeptiert, wenn sie wörtlich (in deutscher Schreibweise) im Umfeld
    der angegebenen Seite abgedruckt ist. Kleine Ganzzahlen (Alter, Faktoren)
    gelten immer als belegt."""
    if abs(wert) < 1000 and float(wert).is_integer():
        return True
    texte = [s["flaeche"] for s in dok.seiten
             if seite and abs(s["nummer"] - seite) <= umfeld]
    if not texte:
        texte = [s["flaeche"] for s in dok.seiten]
    kandidaten = {f"{wert:.{dez}f}".replace(".", ",") for dez in range(7)}
    return any(k in t for t in texte for k in kandidaten)


@check("L-02", "Nachrechnen der Beispiel- und Formelwerte", "dokumentintern", "LLM")
def check_beispielrechnungen(dok):
    """Claude extrahiert Formeln samt eingesetzten Zahlen und behaupteten
    Ergebnissen aus dem PDF; das Notebook rechnet jede Formel in Python nach
    (sichere AST-Auswertung) und vergleicht mit dem abgedruckten Wert."""
    schema = _schema_liste("rechnungen", {
        "beschreibung": {"type": "string"},
        "formel": {"type": "string",
                   "description": "Python-Ausdruck, nur + - * / ** und Klammern"},
        "eingaben": {"type": "array", "items": {
            "type": "object",
            "properties": {"name": {"type": "string"}, "wert": {"type": "number"}},
            "required": ["name", "wert"], "additionalProperties": False}},
        "ergebnis": {"type": "number"},
        "nachkommastellen": {"type": "integer"},
        "seite": {"type": "integer"}},
        ["beschreibung", "formel", "eingaben", "ergebnis", "nachkommastellen", "seite"])
    auftrag = (
        "Extrahiere aus dem folgenden TB-Auszug ALLE nachrechenbaren Rechnungen: "
        "Stellen, an denen eine Formel mit konkreten Zahlen ausgewertet und ein "
        "Ergebnis behauptet wird (typisch: Beispielrechnungen, eingesetzte "
        "Gleichungen).\n"
        "- 'formel': als Python-Ausdruck mit Variablennamen [A-Za-z_][A-Za-z0-9_]*; "
        "erlaubt sind nur + - * / ** und Klammern. Brüche im Formelsatz erscheinen "
        "als 'Zähler ... Nenner' untereinander – setze sie korrekt als Division um.\n"
        "- 'eingaben': alle in der Formel verwendeten Variablen mit den im Text "
        "eingesetzten Zahlwerten (deutsche Zahlen in Dezimalpunkt umwandeln).\n"
        "- 'ergebnis': der im Text behauptete Zahlwert; 'nachkommastellen': dessen "
        "angezeigte Nachkommastellen.\n"
        "- Übernimm nur Rechnungen, bei denen Formel, alle Eingaben und das "
        "Ergebnis eindeutig im Text stehen. Gib die Seite aus dem letzten "
        "[Seite N]-Marker an.")
    # Kandidaten: Seiten mit erkennbarem Formelsatz (mehrere '='-Zeichen)
    kandidaten = [(s["nummer"], s["flaeche"]) for s in dok.seiten
                  if not s["ist_ihv"] and s["flaeche"].count("=") >= 3]
    if not kandidaten:
        raise CheckUebersprungen("Keine Seiten mit Formelsatz erkannt.")
    befunde, n_rechnungen, n_uebersprungen = [], 0, 0
    for chunk in _text_chunks(kandidaten, max_zeichen=9000):
        antwort = llm_json(auftrag, chunk, schema, max_tokens=12000)
        for r in antwort.get("rechnungen", []):
            variablen = {e["name"]: e["wert"] for e in r["eingaben"]}
            # Allgemeine Formeln ohne konkrete Zahlwerte sind nicht nachrechenbar
            try:
                frei = {k.id for k in ast.walk(ast.parse(r["formel"], mode="eval"))
                        if isinstance(k, ast.Name)} - set(_ERLAUBTE_FUNKTIONEN)
            except SyntaxError:
                n_uebersprungen += 1
                continue
            if not frei <= set(variablen):
                n_uebersprungen += 1
                continue
            # Grounding: Eingaben und behauptetes Ergebnis müssen wörtlich im
            # Dokument stehen, sonst ist die Extraktion nicht belastbar.
            if not all(_zahl_steht_im_dokument(dok, w, r.get("seite"))
                       for w in [*variablen.values(), r["ergebnis"]]):
                n_uebersprungen += 1
                continue
            n_rechnungen += 1
            try:
                berechnet = sichere_formelauswertung(r["formel"], variablen)
            except Exception as fehler:
                befunde.append(Befund(
                    f"Rechnung „{r['beschreibung']}“ konnte nicht ausgewertet "
                    f"werden ({fehler}) – Formel: {r['formel']}.",
                    r.get("seite"), schwere=WARNUNG))
                continue
            toleranz = 0.51 * 10 ** (-r["nachkommastellen"])
            abweichung = abs(berechnet - r["ergebnis"])
            if abweichung > 25 * toleranz:
                befunde.append(Befund(
                    f"„{r['beschreibung']}“: Das Dokument behauptet "
                    f"{fmt_de(r['ergebnis'], r['nachkommastellen'])}, die Formel "
                    f"{r['formel']} ergibt mit den angegebenen Eingaben aber "
                    f"{fmt_de(berechnet, max(2, r['nachkommastellen']))}.",
                    r.get("seite")))
            elif abweichung > toleranz:
                befunde.append(Befund(
                    f"„{r['beschreibung']}“: Ergebnis "
                    f"{fmt_de(r['ergebnis'], r['nachkommastellen'])} weicht leicht "
                    f"vom nachgerechneten Wert {fmt_de(berechnet, 6)} ab – kann an "
                    f"der Rundungskaskade liegen.", r.get("seite"), schwere=WARNUNG))
    return befunde, (f"{n_rechnungen} Rechnungen in Python nachgerechnet und "
                     f"verglichen; {n_uebersprungen} übersprungen (keine konkreten "
                     f"oder nicht im Dokument belegte Zahlen) ({MODELL}).")


# ---------------------------------------------------------------- L-03
def _teil_text(dok, praefix, nummer, nur_prosa=True):
    """Text eines Teils (von seiner Überschrift bis zur nächsten)."""
    u = dok.ueberschrift(praefix, nummer)
    if u is None:
        return None
    folgende = sorted(x.seite for x in dok.ueberschriften if x.seite > u.seite)
    ende = folgende[0] - 1 if folgende else dok.seitenzahl
    seiten = []
    for s in dok.seiten:
        if u.seite <= s["nummer"] <= ende:
            if nur_prosa:
                zeilen = [z for z in s["zeilen"]
                          if sum(c.isalpha() for c in z) >= max(6, 0.4 * len(z))
                          or (len(z.strip()) <= 40 and re.search(r"%|EUR", z))
                          # Formelzeilen behalten (Vorzeichen-/Formelvergleich!),
                          # reine Zahlenkolonnen der Tabellen bleiben draußen
                          or (len(z.strip()) <= 100 and re.search(r"[=·±]", z))]
                seiten.append((s["nummer"], zeilen_zu_flaeche("\n".join(zeilen))))
            else:
                seiten.append((s["nummer"], s["flaeche"]))
    return "\n".join(f"[Seite {n}] {t}" for n, t in seiten if t.strip())


@check("L-03", "Konsistenz Abschnitt ↔ Anlage/Anhang", "dokumentintern", "LLM")
def check_konsistenz(dok):
    """Abschnitte und ihre gleichnummerigen Anlagen (sowie die
    Beispielrechnung) müssen dieselben Werte, Parameter und Formeln nennen.
    Claude vergleicht die Textpaare; Befunde werden als Warnung zur
    menschlichen Sichtung gemeldet."""
    schema = _schema_liste("widersprueche", {
        "thema": {"type": "string"},
        "stelle_a": {"type": "string"}, "wert_a": {"type": "string"},
        "stelle_b": {"type": "string"}, "wert_b": {"type": "string"},
        "begruendung": {"type": "string"}, "seite": {"type": "integer"}},
        ["thema", "stelle_a", "wert_a", "stelle_b", "wert_b", "begruendung", "seite"])
    auftrag = (
        "Vergleiche die beiden folgenden Auszüge derselben TB (Teil A und "
        "Teil B). Melde AUSSCHLIESSLICH echte inhaltliche Widersprüche: "
        "dieselbe Größe (Parameter, Prozentsatz, Betrag, Zinssatz, Formel, "
        "Vorzeichen) mit unterschiedlichen Werten oder unterschiedlicher "
        "Struktur. Achte besonders auf Vorzeichen in Formeln und darauf, ob "
        "eingesetzte Zahlen zur genannten Formel passen.\n"
        "KEINE Widersprüche sind: unterschiedlicher Detailgrad, Rundungen, "
        "andere Formulierungen, Auslassungen (etwas wird nur in einem Teil "
        "erwähnt), eine Spanne/Klasse im einen und ein konkreter Wert "
        "innerhalb dieser Spanne im anderen Teil, unterschiedliche "
        "Gleichungs-/Verweisnummern sowie alles, was du mangels Kontext "
        "nicht prüfen kannst. Gib die Seite des Widerspruchs aus dem "
        "nächstgelegenen [Seite N]-Marker an.")
    # Paare: Abschnitt N.N <-> Anlage N.N (gleiche Nummer) + Abschnitt 5.1 <-> Anhänge
    paare = []
    for u in dok.ueberschriften:
        if u.praefix != "Anlage":
            continue
        if dok.ueberschrift("Abschnitt", u.nummer):
            paare.append((("Abschnitt", u.nummer), ("Anlage", u.nummer)))
    for anhang in [u for u in dok.ueberschriften if u.praefix == "Anhang"]:
        for abschnitt in ("5.1", "3.2"):
            if dok.ueberschrift("Abschnitt", abschnitt) \
                    and re.search(r"beispiel", anhang.titel, re.I):
                paare.append((("Abschnitt", abschnitt), ("Anhang", anhang.nummer)))
    if not paare:
        raise CheckUebersprungen("Keine Abschnitt/Anlage-Paare gefunden.")
    befunde = []
    gesehen = set()
    for (pa, na), (pb, nb) in paare:
        text_a = _teil_text(dok, pa, na)
        text_b = _teil_text(dok, pb, nb)
        if not text_a or not text_b:
            continue
        text = (f"### Teil A: {pa} {na}\n{text_a[:6000]}\n\n"
                f"### Teil B: {pb} {nb}\n{text_b[:6000]}")
        antwort = llm_json(auftrag, text, schema, max_tokens=16000)
        for w in antwort.get("widersprueche", []):
            kennung = (w["thema"].casefold()[:40], w["wert_a"], w["wert_b"])
            if kennung in gesehen:
                continue
            gesehen.add(kennung)
            befunde.append(Befund(
                f"Widerspruch „{w['thema']}“: {w['stelle_a']} nennt "
                f"„{w['wert_a']}“, {w['stelle_b']} dagegen „{w['wert_b']}“ – "
                f"{w['begruendung']}", w.get("seite"), schwere=WARNUNG))
    return befunde, f"{len(paare)} Teil-Paare mit {MODELL} verglichen."


# ---------------------------------------------------------------- L-04
# Werkzeug „gesetzestext“: amtlicher Wortlaut eines Paragrafen, live von
# gesetze-im-internet.de (Bundesministerium der Justiz). Claude entscheidet
# selbst, welche Paragrafen es nachschlägt. Ist die Seite nicht erreichbar,
# dient die gespeicherte Kopie (NORMTEXT_KOPIE) als Ersatz.
import html as _html
import threading
import urllib.error
import urllib.request
from datetime import date

_GESETZE_IM_INTERNET = {   # Kürzel in der TB -> Pfad auf gesetze-im-internet.de
    "VAG": "vag_2016", "KVAV": "kvav", "VVG": "vvg_2008", "HGB": "hgb",
    "RechVersV": "rechversv", "SGB V": "sgb_5", "SGB XI": "sgb_11", "BGB": "bgb"}
_normtext_sperre = threading.Lock()
_normtext_stand = {"fehlschlaege": 0, "kopie": None, "abgerufen": {}, "takt": 0.0}
# gesetze-im-internet.de trennt bei schnellen Abrufserien die Verbindung:
# höchstens zwei Abrufe gleichzeitig, mindestens 0,15 s Abstand, drei Versuche.
_NORMTEXT_GLEICHZEITIG = threading.Semaphore(2)
_NORMTEXT_ABSTAND_S = 0.15


def _normtext_laden(url: str) -> bytes:
    for versuch in range(3):
        with _NORMTEXT_GLEICHZEITIG:
            with _normtext_sperre:
                termin = max(time.time(), _normtext_stand["takt"] + _NORMTEXT_ABSTAND_S)
                _normtext_stand["takt"] = termin
            time.sleep(max(0.0, termin - time.time()))
            try:
                anfrage = urllib.request.Request(
                    url, headers={"User-Agent": "TB-Validierung (DAV-Arbeitsgruppe)"})
                with urllib.request.urlopen(anfrage, timeout=5) as antwort:
                    return antwort.read()
            except urllib.error.HTTPError as fehler:
                # 404 = gibt es nicht; 403/407 u. Ä. (z. B. Firmen-Proxy): sofort aufgeben
                if fehler.code not in (429, 500, 502, 503, 504) or versuch == 2:
                    raise
            except (urllib.error.URLError, OSError) as fehler:
                # Nur das Trennen bei Abrufserien lohnt einen neuen Versuch; nicht
                # erreichbar oder gesperrt -> gleich die gespeicherte Kopie
                grund = getattr(fehler, "reason", fehler)
                if not isinstance(grund, (ConnectionResetError, ConnectionAbortedError)) or versuch == 2:
                    raise
        time.sleep(0.8 * (versuch + 1))


def _normtext_aus_html(roh: bytes) -> str:
    """Überschrift und Absätze einer Paragrafenseite als Text (Absätze und
    Nummern auf eigenen Zeilen)."""
    seite = roh.decode("iso-8859-1")
    kopf = " ".join(_html.unescape(re.sub(r"<[^>]+>", "", m)) for m in re.findall(
        r'<span class="jn(?:enbez|entitel)">(.*?)</span>', seite, re.S))
    anfang = seite.find('<div class="jnhtml">')
    ende = seite.find('<div id="fusszeile"', max(anfang, 0))
    if anfang < 0:
        raise ValueError("Seite enthält keinen Normtext.")
    rumpf = seite[anfang:ende if ende > 0 else None]
    rumpf = re.sub(r"<h4>(.*?)</h4>", r"\n\1:", rumpf, flags=re.S)   # „Fußnote:“
    rumpf = re.sub(r'<div class="jurAbsatz">', "\n", rumpf)
    rumpf = re.sub(r"<dt>", "\n  ", rumpf)
    rumpf = re.sub(r"<[^>]+>", " ", rumpf)
    zeilen = [re.sub(r"[ \t ]+", " ", _html.unescape(z)).strip()
              for z in rumpf.split("\n")]
    text = "\n".join(z for z in zeilen if z)
    if not text:
        raise ValueError("Seite enthält keinen Normtext.")
    return f"{kopf.strip()}\n{text}"


def _normtext_kopie() -> dict:
    if _normtext_stand["kopie"] is None:
        try:
            _normtext_stand["kopie"] = json.loads(
                Path(NORMTEXT_KOPIE).read_text(encoding="utf-8"))
        except (FileNotFoundError, TypeError, json.JSONDecodeError):
            _normtext_stand["kopie"] = {}
    return _normtext_stand["kopie"]


def gesetzestext(gesetz: str, paragraf: str) -> str:
    """Wortlaut von „§ <paragraf> <gesetz>“ samt Quelle (Werkzeug für L-04)."""
    gesetz = gesetz.strip()
    paragraf = paragraf.replace("§", "").strip().lower()
    if gesetz not in _GESETZE_IM_INTERNET:
        raise ValueError(f"Gesetz „{gesetz}“ ist nicht verfügbar "
                         f"(verfügbar: {', '.join(_GESETZE_IM_INTERNET)}).")
    if not re.fullmatch(r"\d{1,4}[a-z]?", paragraf):
        raise ValueError(f"„{paragraf}“ ist keine Paragrafennummer (Beispiel: 8 oder 341f).")
    schluessel = f"§ {paragraf} {gesetz}"
    with _normtext_sperre:
        if schluessel in _normtext_stand["abgerufen"]:
            return _normtext_stand["abgerufen"][schluessel]
        # nach einem endgültig gescheiterten Abruf: Rest des Laufs aus der Kopie
        online = _normtext_stand["fehlschlaege"] == 0 and bool(NORMTEXT_QUELLE)
    url = f"https://www.gesetze-im-internet.de/{_GESETZE_IM_INTERNET[gesetz]}/__{paragraf}.html"
    text = None
    if online:
        try:
            text = _normtext_aus_html(_normtext_laden(
                f"{str(NORMTEXT_QUELLE).rstrip('/')}/{_GESETZE_IM_INTERNET[gesetz]}/__{paragraf}.html"))
            quelle = f"{url}, abgerufen am {date.today():%d.%m.%Y}"
        except urllib.error.HTTPError as fehler:
            if fehler.code == 404:
                raise ValueError(f"{schluessel} gibt es nicht (gesetze-im-internet.de).") from None
            with _normtext_sperre:
                _normtext_stand["fehlschlaege"] += 1
        except (urllib.error.URLError, OSError, ValueError):
            with _normtext_sperre:
                _normtext_stand["fehlschlaege"] += 1
    if text is None:
        eintrag = _normtext_kopie().get(schluessel)
        if not eintrag:
            raise ValueError(f"{schluessel}: gesetze-im-internet.de nicht erreichbar und "
                             f"keine gespeicherte Kopie vorhanden.")
        text, quelle = eintrag["text"], f"{eintrag['url']}, gespeicherte Kopie vom {eintrag['stand']}"
    elif NORMTEXT_KOPIE_AKTUALISIEREN:
        with _normtext_sperre:
            kopie = _normtext_kopie()
            kopie[schluessel] = {"url": url, "stand": f"{date.today():%d.%m.%Y}", "text": text}
            Path(NORMTEXT_KOPIE).write_text(
                json.dumps(dict(sorted(kopie.items())), ensure_ascii=False, indent=1),
                encoding="utf-8")
    ergebnis = f"Quelle: {quelle}\n{text}"
    with _normtext_sperre:
        _normtext_stand["abgerufen"][schluessel] = ergebnis
    return ergebnis


GESETZESTEXT_WERKZEUG = Werkzeug(
    name="gesetzestext",
    beschreibung=(
        "Liefert den amtlichen Wortlaut eines Paragrafen mit allen Absätzen, Sätzen "
        "und Nummern (Quelle: gesetze-im-internet.de). Verfügbar: "
        + ", ".join(_GESETZE_IM_INTERNET) + ". Für jede Vorschrift, die du beurteilst "
        "oder als einschlägig nennen willst, vorher aufrufen; mehrere Aufrufe "
        "gleichzeitig sind erwünscht."),
    eingabe_schema={
        "type": "object",
        "properties": {
            "gesetz": {"type": "string", "enum": list(_GESETZE_IM_INTERNET)},
            "paragraf": {"type": "string",
                         "description": "Paragrafennummer ohne §, z. B. 8 oder 341f"}},
        "required": ["gesetz", "paragraf"], "additionalProperties": False},
    ausfuehren=gesetzestext)


@check("L-04", "Plausibilität der Normzitate", "dokumentextern", "LLM")
def check_normzitate_llm(dok):
    """Claude prüft jedes Normzitat am amtlichen Wortlaut: Es schlägt die
    zitierten (und ggf. die eigentlich einschlägigen) Paragrafen per
    Werkzeugaufruf auf gesetze-im-internet.de nach und urteilt danach."""
    if not dok.paragrafen:
        raise CheckUebersprungen("Keine Normzitate im Dokument gefunden.")
    with _normtext_sperre:
        _normtext_stand["fehlschlaege"] = 0    # neuer Lauf: wieder live nachschlagen
        _normtext_stand["abgerufen"] = {}
    schema = _schema_liste("bewertungen", {
        "zitat": {"type": "string"},
        "bewertung": {"type": "string", "enum": [
            "passt", "fundstelle_abweichend", "mittelbar", "falsch", "nicht_pruefbar"]},
        "korrekte_norm": {"type": "string"},
        "begruendung": {"type": "string"},
        "seite": {"type": "integer"}},
        ["zitat", "bewertung", "korrekte_norm", "begruendung", "seite"])
    auftrag = (
        "Unten stehen Normzitate aus einer TB der deutschen PKV mit ihrem Kontext. "
        "Prüfe jedes Zitat am amtlichen Wortlaut: Regelt die zitierte Stelle "
        "(Paragraf, Absatz, Satz, Nummer) den beschriebenen Sachverhalt?\n"
        "Vorgehen: Schlage mit dem Werkzeug gesetzestext jeden zitierten Paragrafen "
        "nach, alle Abrufe möglichst gleichzeitig in einer Antwort. Hältst du eine "
        "andere Vorschrift für einschlägig, schlage auch sie nach, bevor du sie nennst. "
        "Urteile nach dem abgerufenen Wortlaut, nicht aus dem Gedächtnis.\n"
        "Bewertung:\n"
        "- 'passt': Der Wortlaut der zitierten Stelle trägt den Sachverhalt (bei "
        "einem Zitat ohne Absatz genügt der Paragraf).\n"
        "- 'fundstelle_abweichend': richtiger Paragraf, aber der Sachverhalt steht "
        "laut Wortlaut in einem anderen Absatz, Satz oder einer anderen Nummer; "
        "nenne die richtige Fundstelle in 'korrekte_norm'.\n"
        "- 'mittelbar': Der Paragraf gehört zum selben Sachgebiet wie die Aussage "
        "(z. B. Prämienkalkulation, Alterungsrückstellung, Tarifwechsel), sein Wortlaut "
        "trägt die konkrete Aussage aber nicht ausdrücklich – etwa weil sie nur daraus "
        "abgeleitet ist.\n"
        "- 'falsch': Der zitierte Paragraf gehört laut Wortlaut zu einem anderen "
        "Sachgebiet (z. B. Aufsichtsverfahren, Unternehmensorganisation, Vertriebs- oder "
        "Steuerrecht statt Kalkulation); nenne in 'korrekte_norm' die einschlägige "
        "Vorschrift, wenn du sie nachgeschlagen hast, sonst lass das Feld leer.\n"
        "- 'nicht_pruefbar': Der Wortlaut war nicht abrufbar.\n"
        "- Vertragsklauseln wie '§ 8b AVB' oder 'MB/KK' sind keine Gesetzeszitate: "
        "'passt'.\n"
        "'begruendung': höchstens zwei sachliche Sätze mit dem entscheidenden Inhalt des "
        "Wortlauts, ohne Angaben zu deinem Vorgehen.")
    eintraege = []
    gesehen = set()
    for p in dok.paragrafen:
        kennung = (p["zitat"], p["kontext"][:60])
        if kennung in gesehen:
            continue
        gesehen.add(kennung)
        eintraege.append((p["seite"], f"{p['zitat']} | Kontext: ...{p['kontext']}..."))
    befunde = []
    n = 0
    # Kleinere Pakete laufen (in der Demo-Oberfläche) parallel und damit schneller.
    for chunk in _text_chunks(eintraege, max_zeichen=4500):
        antwort = llm_json(auftrag, chunk, schema, max_tokens=12000,
                           werkzeuge=[GESETZESTEXT_WERKZEUG])
        for b in antwort.get("bewertungen", []):
            n += 1
            # nur das Zitat selbst (manche Antworten hängen eine Umschreibung in Klammern an)
            b["zitat"] = re.sub(r"\s*\(.*$", "", b["zitat"]).strip() or b["zitat"]
            korrekt = b["korrekte_norm"].strip()
            if b["bewertung"] == "falsch":
                einschlaegig = f" – einschlägig wäre {korrekt}" if korrekt else ""
                befunde.append(Befund(
                    f"Normzitat „{b['zitat']}“ passt nicht zum Sachverhalt{einschlaegig}. "
                    f"({b['begruendung']})", b.get("seite")))
            elif b["bewertung"] == "fundstelle_abweichend":
                fundstelle = f" Laut Gesetzestext steht der Sachverhalt in {korrekt}." if korrekt else ""
                befunde.append(Befund(
                    f"Normzitat „{b['zitat']}“: richtiger Paragraf, aber andere "
                    f"Fundstelle.{fundstelle} ({b['begruendung']})",
                    b.get("seite"), schwere=WARNUNG))
            elif b["bewertung"] == "mittelbar":
                befunde.append(Befund(
                    f"Normzitat „{b['zitat']}“: Der Gesetzestext trägt die Aussage nicht "
                    f"ausdrücklich. ({b['begruendung']})", b.get("seite"), schwere=WARNUNG))
            elif b["bewertung"] == "nicht_pruefbar":
                befunde.append(Befund(
                    f"Normzitat „{b['zitat']}“ konnte nicht am Gesetzestext geprüft "
                    f"werden: {b['begruendung']}", b.get("seite"), schwere=WARNUNG))
    return befunde, (f"{n} Normzitate mit {MODELL} am amtlichen Wortlaut geprüft "
                     f"(nachgeschlagen per Werkzeugaufruf auf gesetze-im-internet.de).")


# ---------------------------------------------------------------- L-05
@check("L-05", "Vision-Prüfung des Formelsatzes", "dokumentintern", "LLM")
def check_formeln_vision(dok):
    """Der PDF-Text-Layer zerlegt gesetzte Formeln in Fragmente – Vorzeichen-
    und Strukturfehler sind dort oft unsichtbar. Dieser Check zeigt Claude
    jede formelhaltige Seite deshalb als BILD: Das Modell liest die Formeln
    im Originalsatz und meldet Widersprüche zwischen Formel und zugehöriger
    Beschreibung sowie offensichtliche Vorzeichen-/Strukturfehler."""
    schema = _schema_liste("befunde", {
        "formel": {"type": "string"},
        "problem": {"type": "string"},
        "begruendung": {"type": "string"},
        "eindeutig": {"type": "boolean"}},
        ["formel", "problem", "begruendung", "eindeutig"])
    auftrag = (
        "Das Bild zeigt eine Seite Technischer Berechnungsgrundlagen der PKV. "
        "Prüfe ausschließlich die dort abgedruckten mathematischen Formeln:\n"
        "1. Widerspricht eine Formel ihrer eigenen Legende oder dem "
        "umgebenden Text? (Beispiel: die Formel addiert einen Term, der "
        "Text spricht vom Abzug desselben Terms.)\n"
        "2. Enthält eine Formel einen offensichtlichen mathematischen Fehler "
        "(falsches Vorzeichen, vertauschter Zähler/Nenner, falscher Index)?\n"
        "Melde NUR solche Widersprüche und Fehler. KEINE Befunde sind: "
        "stilistische Anmerkungen, Rundungsdifferenzen, fehlende oder "
        "verkürzte Herleitungen, didaktische Vereinfachungen, Spezialfälle "
        "mit bewusst anderen Formeln (z. B. Kinder/Jugendliche ohne einzelne "
        "Terme), Umformungen zwischen äquivalenten Darstellungen sowie alles, "
        "was Inhalte anderer Seiten voraussetzt. Ein meldewürdiger "
        "Widerspruch betrifft DENSELBEN Term an DERSELBEN Stelle mit "
        "entgegengesetztem Vorzeichen bzw. entgegengesetzter Operation. Setze "
        "'eindeutig' nur auf true, wenn der Widerspruch zweifelsfrei auf "
        "dieser Seite sichtbar ist.")
    kandidaten = [s for s in dok.seiten
                  if not s["ist_ihv"] and s["flaeche"].count("=") >= 3]
    if not kandidaten:
        raise CheckUebersprungen("Keine Seiten mit Formelsatz erkannt.")
    befunde = []
    for s in kandidaten:
        antwort = llm_json(auftrag, f"[Seite {s['nummer']}]", schema,
                           max_tokens=4000,
                           bild_b64=dok.seite_als_png_b64(s["nummer"]))
        for b in antwort.get("befunde", []):
            befunde.append(Befund(
                f"Formel „{b['formel']}“: {b['problem']} – {b['begruendung']}",
                s["nummer"],
                schwere=FEHLER if b.get("eindeutig") else WARNUNG))
    return befunde, f"{len(kandidaten)} Formelseiten als Bild geprüft ({MODELL})."




# ---------------------------------------------------------------- L-06
def _kurven_kandidaten(dok, seite: int, umfeld: int = 6):
    """Alle numerischen Tabellenspalten im Umfeld einer Grafikseite als
    Vergleichskandidaten: (beschriftung, geschlecht, {alter: wert})."""
    kandidaten = []
    for t in dok.tabellen:
        if not t.spalten or t.spalten[0] != "x":
            continue
        if not any(abs(s - seite) <= umfeld for s in t.seiten):
            continue
        for idx in range(1, len(t.spalten)):
            werte = {}
            for alter in t.zeilen:
                try:
                    werte[alter] = t.wert(alter, idx)
                except (ValueError, IndexError):
                    continue
            if len(werte) < 10:
                continue
            ges = (t.geschlecht_je_spalte[idx]
                   if idx < len(t.geschlecht_je_spalte) else None) or t.geschlecht
            kandidaten.append(
                (f"{t.kurzname()}, Spalte „{t.spalten[idx]}“ (Nr. {idx + 1})",
                 ges, werte))
    return kandidaten


def _median(werte):
    s = sorted(werte)
    n = len(s)
    return s[n // 2] if n % 2 else 0.5 * (s[n // 2 - 1] + s[n // 2])


@check("L-06", "Grafik-Tabellen-Abgleich", "dokumentintern", "LLM")
def check_grafik_tabellen(dok):
    """Diagramme müssen zu den abgedruckten Tabellenwerten passen. Damit die
    Ablesung nicht von den Tabellenwerten beeinflusst wird (Verankerungs-
    effekt: ein Modell, das Soll-Werte kennt, „bestätigt“ sie), liest Claude
    die Kurven BLIND ab – es bekommt nur das Seitenbild, keine Tabellenwerte.
    Der Vergleich erfolgt anschließend deterministisch in Python: Jede Kurve
    wird der am besten passenden Tabellenspalte des Seitenumfelds zugeordnet
    (bei erkennbarem Geschlecht der Kurve nur Spalten desselben Geschlechts);
    weicht selbst die beste Spalte im Median um mehr als 20 % ab, liegt ein
    krasser Widerspruch vor. Feinere Abweichungen liegen innerhalb der
    Ablesegenauigkeit und werden nicht gemeldet."""
    grafik_seiten = dok.grafik_seiten()
    if not grafik_seiten:
        raise CheckUebersprungen("Keine Seiten mit Diagrammen erkannt.")
    schema = {
        "type": "object",
        "properties": {
            "kurven": {"type": "array", "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "punkte": {"type": "array", "items": {
                        "type": "object",
                        "properties": {"x": {"type": "number"},
                                       "y": {"type": "number"}},
                        "required": ["x", "y"],
                        "additionalProperties": False}}},
                "required": ["name", "punkte"],
                "additionalProperties": False}},
            "ablesbarkeit": {"type": "string"},
        },
        "required": ["kurven", "ablesbarkeit"],
        "additionalProperties": False,
    }
    auftrag = (
        "Das Bild zeigt eine Seite einer TB mit einem Diagramm.\n"
        "Lies die Kurven UNABHÄNGIG und ausschließlich aus dem Diagramm ab. "
        "Wichtig: Nutze KEINE Tabellen- oder Textwerte, auch wenn auf der "
        "Seite Tabellen zu sehen sind – miss die Kurvenhöhe allein anhand "
        "der Achsenbeschriftung und der Gitterlinien.\n"
        "1. Benenne jede Kurve gemäß Legende ('name').\n"
        "2. Gib je Kurve die abgelesenen Werte an 5–8 über den gesamten "
        "Definitionsbereich verteilten, gut ablesbaren Stützstellen an "
        "('punkte': x = Wert auf der x-Achse, y = abgelesener Kurvenwert). "
        "Bevorzuge runde x-Werte (z. B. 30, 40, 60, 80, 100).\n"
        "3. Beschreibe in 'ablesbarkeit' kurz Achsenskala, Auflösung und "
        "Grenzen der Ablesung.")
    befunde = []
    ablesbarkeiten = []
    n_kurven = 0
    for seite in grafik_seiten:
        kandidaten = _kurven_kandidaten(dok, seite)
        if not kandidaten:
            continue
        antwort = llm_json(auftrag, f"[Seite {seite}]", schema,
                           max_tokens=4000,
                           bild_b64=dok.seite_als_png_b64(seite, dpi=150))
        ablesbarkeiten.append(f"S.{seite}: {antwort.get('ablesbarkeit', '?')}")
        for kurve in antwort.get("kurven", []):
            name = kurve.get("name", "?")
            punkte = {int(round(p["x"])): float(p["y"])
                      for p in kurve.get("punkte", [])}
            if len(punkte) < 3:
                continue
            ges_kurve = ("M" if re.search(r"männ|herren", name, re.I) else
                         "F" if re.search(r"frau|weib|damen", name, re.I) else None)
            # Beste Tabellenspalte: minimale mediane relative Abweichung an
            # den abgelesenen Stützstellen (Geschlecht schränkt ein).
            beste = None
            for beschriftung, ges_spalte, werte in kandidaten:
                if ges_kurve and ges_spalte and ges_spalte != ges_kurve:
                    continue
                abw = [(abs(y - werte[x]) / abs(werte[x]), x, y, werte[x])
                       for x, y in punkte.items()
                       if x in werte and werte[x] != 0]
                if len(abw) < 3:
                    continue
                med = _median([a[0] for a in abw])
                if beste is None or med < beste[0]:
                    beste = (med, beschriftung, abw)
            if beste is None:
                continue
            n_kurven += 1
            med, beschriftung, abw = beste
            if med > 0.20:
                beispiele = "; ".join(
                    f"x={x}: Grafik ≈{fmt_de(y, 2)}, Tabelle {fmt_de(w, 4)}"
                    for _, x, y, w in sorted(abw, reverse=True)[:3])
                befunde.append(Befund(
                    f"Grafik widerspricht den Tabellenwerten – Kurve "
                    f"„{name}“ weicht selbst von der am besten passenden "
                    f"Spalte ({beschriftung}) im Median um "
                    f"{med * 100:.0f} % ab ({beispiele}).", seite))
    hinweis = (f"{len(grafik_seiten)} Diagrammseite(n) blind abgelesen "
               f"({MODELL}), {n_kurven} Kurven deterministisch den "
               f"Tabellenspalten zugeordnet (krass = Median-Abweichung "
               f"> 20 %). Ablesegenauigkeit: " + " | ".join(ablesbarkeiten))
    return befunde, hinweis[:700]


print(f"{sum(1 for c in CHECKS if c['methode'] == 'LLM')} LLM-Checks registriert "
      f"(Modell: {MODELL}).")

6 LLM-Checks registriert (Modell: claude-sonnet-5).

## 8 | Ausführung

Alle registrierten Checks laufen nacheinander; Abstürze einzelner Checks
werden abgefangen, damit der Lauf immer vollständig durchläuft.

In [8]:
# =============================================================================
# Ausführung aller registrierten Checks
# =============================================================================
ergebnisse = []
for eintrag in CHECKS:
    start = time.time()
    ergebnis = Pruefergebnis(eintrag["id"], eintrag["titel"], eintrag["kategorie"],
                             eintrag["methode"])
    try:
        befunde, hinweis = eintrag["fn"](dok)
        ergebnis.befunde = befunde
        ergebnis.hinweis = hinweis
        if any(b.schwere == FEHLER for b in befunde):
            ergebnis.status = FEHLER
        elif befunde:
            ergebnis.status = WARNUNG
        else:
            ergebnis.status = BESTANDEN
    except CheckUebersprungen as grund:
        ergebnis.status = UEBERSPRUNGEN
        ergebnis.hinweis = str(grund)
    except Exception as fehler:  # Check-Absturz soll den Lauf nicht stoppen
        ergebnis.status = FEHLGESCHLAGEN
        ergebnis.hinweis = f"{type(fehler).__name__}: {fehler}"
    ergebnis.dauer_s = round(time.time() - start, 2)
    ergebnisse.append(ergebnis)
    symbol = {BESTANDEN: "[OK]    ", WARNUNG: "[WARN]  ", FEHLER: "[FEHLER]",
              UEBERSPRUNGEN: "[SKIP]  ", FEHLGESCHLAGEN: "[CRASH] "}[ergebnis.status]
    print(f"{symbol} {ergebnis.check_id} {ergebnis.titel} "
          f"({len(ergebnis.befunde)} Befunde, {ergebnis.dauer_s}s)")
    if ergebnis.status in (UEBERSPRUNGEN, FEHLGESCHLAGEN):
        print(f"         -> {ergebnis.hinweis}")

[FEHLER] V-01 Kaputte Verweise („??“) (1 Befunde, 0.0s)
[OK]     V-02 Existenz interner Verweise (0 Befunde, 0.0s)
[FEHLER] V-03 Inhaltsverzeichnis (Titel und Seitenzahlen) (1 Befunde, 0.0s)
[FEHLER] V-04 Vollständigkeit des Symbolverzeichnisses (1 Befunde, 0.01s)
[FEHLER] V-05 Tabellen-Identitäten und Rechnungszins (2 Befunde, 0.0s)
[WARN]   V-06 Monotonie der Beitragstabellen (1 Befunde, 0.0s)
[FEHLER] V-07 Abgleich mit Referenzwerten (CSV) (2 Befunde, 0.07s)
[FEHLER] V-08 Konsistenz benannter Parameter (1 Befunde, 0.0s)
[FEHLER] V-09 Gesetzes- und Paragrafen-Prüfung (3 Befunde, 0.0s)
[OK]     V-10 Referenzen auf Rahmen-Berechnungsgrundlagen (0 Befunde, 0.0s)
[OK]     V-11 Versions- und Stichtagskonsistenz (0 Befunde, 0.0s)
[OK]     V-12 Vollständigkeit und Wertebereiche der Alterstabellen (0 Befunde, 0.0s)
[FEHLER] V-13 Abgleich mit der Referenzdatenbank (SQLite) (1 Befunde, 0.0s)
[FEHLER] L-01 Rechtschreibprüfung (3 Befunde, 0.01s)
[FEHLER] L-02 Nachrechnen der Beispiel- und Formel

## 9 | Prüfbericht

Übersicht je Check (Ampelfarben), alle Befunde im Detail sowie ein
JSON-Export für die Weiterverarbeitung.

In [9]:
# =============================================================================
# Strukturierter Prüfbericht
# =============================================================================

# --- Übersicht je Check --------------------------------------------------------
uebersicht = pd.DataFrame([{
    "Check": e.check_id, "Titel": e.titel, "Kategorie": e.kategorie,
    "Methode": e.methode, "Status": e.status,
    "Fehler": sum(1 for b in e.befunde if b.schwere == FEHLER),
    "Warnungen": sum(1 for b in e.befunde if b.schwere == WARNUNG),
    "Umfang / Hinweis": e.hinweis,
} for e in ergebnisse])

_FARBEN = {BESTANDEN: "#e8f5e9", WARNUNG: "#fff8e1", FEHLER: "#ffebee",
           UEBERSPRUNGEN: "#eceff1", FEHLGESCHLAGEN: "#ede7f6"}


def _einfaerben(zeile):
    return [f"background-color: {_FARBEN.get(zeile['Status'], 'white')}"] * len(zeile)


from IPython.display import display

print(f"Prüfbericht für: {Path(TB_PDF_PFAD).name}")
display(uebersicht.style.apply(_einfaerben, axis=1).hide(axis="index"))

# --- Alle Befunde im Detail ----------------------------------------------------
befund_zeilen = [{
    "Check": e.check_id, "Schwere": b.schwere, "Seite": b.seite,
    "Befund": b.beschreibung,
} for e in ergebnisse for b in e.befunde]
befunde_df = pd.DataFrame(befund_zeilen,
                          columns=["Check", "Schwere", "Seite", "Befund"])
if len(befunde_df):
    with pd.option_context("display.max_colwidth", None):
        display(befunde_df.style.apply(
            lambda z: [f"background-color: "
                       f"{_FARBEN[FEHLER] if z['Schwere'] == FEHLER else _FARBEN[WARNUNG]}"] * len(z),
            axis=1).hide(axis="index"))
else:
    print("Keine Befunde – alle ausgeführten Checks bestanden.")

# --- Gesamtfazit und JSON-Export -----------------------------------------------
n_fehler = int((befunde_df["Schwere"] == FEHLER).sum()) if len(befunde_df) else 0
n_warnungen = int((befunde_df["Schwere"] == WARNUNG).sum()) if len(befunde_df) else 0
n_uebersprungen = sum(1 for e in ergebnisse if e.status == UEBERSPRUNGEN)

print()
print("=" * 72)
print(f"GESAMTERGEBNIS: {n_fehler} Fehler, {n_warnungen} Warnungen "
      f"({len(ergebnisse)} Checks, davon {n_uebersprungen} übersprungen)")
if n_fehler:
    print("=> Das Dokument enthält Befunde, die vor einer Freigabe zu klären sind.")
elif n_warnungen:
    print("=> Keine Fehler; die Warnungen sollten gesichtet werden.")
else:
    print("=> Keine Auffälligkeiten gefunden.")
print("=" * 72)

bericht_pfad = (Path(BERICHT_PFAD) if BERICHT_PFAD
                else Path(f"pruefbericht_{Path(TB_PDF_PFAD).stem}.json"))
bericht = {
    "dokument": str(TB_PDF_PFAD), "seiten": dok.seitenzahl, "modell": MODELL,
    "zusammenfassung": {"fehler": n_fehler, "warnungen": n_warnungen,
                        "checks": len(ergebnisse), "uebersprungen": n_uebersprungen},
    "checks": [{
        "id": e.check_id, "titel": e.titel, "kategorie": e.kategorie,
        "methode": e.methode, "status": e.status, "hinweis": e.hinweis,
        "dauer_s": e.dauer_s,
        "befunde": [{"schwere": b.schwere, "seite": b.seite,
                     "beschreibung": b.beschreibung} for b in e.befunde],
    } for e in ergebnisse],
}
bericht_pfad.write_text(json.dumps(bericht, ensure_ascii=False, indent=2),
                        encoding="utf-8")
print(f"Prüfbericht gespeichert: {bericht_pfad}")

Prüfbericht für: TB_VitalPlus_fehlerhaft.pdf


Check,Titel,Kategorie,Methode,Status,Fehler,Warnungen,Umfang / Hinweis
V-01,Kaputte Verweise („??“),dokumentintern,deterministisch,FEHLER,1,0,93 Seiten durchsucht.
V-02,Existenz interner Verweise,dokumentintern,deterministisch,BESTANDEN,0,0,252 Teil-Verweise und 35 Gleichungs-Verweise geprüft.
V-03,Inhaltsverzeichnis (Titel und Seitenzahlen),dokumentintern,deterministisch,FEHLER,1,0,65 IHV-Einträge gegen Überschriften geprüft (Seitenversatz +1).
V-04,Vollständigkeit des Symbolverzeichnisses,dokumentintern,deterministisch,FEHLER,1,0,141 verschiedene Symbole gegen 58 Glossar-Einträge geprüft.
V-05,Tabellen-Identitäten und Rechnungszins,dokumentintern,deterministisch,FEHLER,2,0,180 Zeilen-Identitäten geprüft.
V-06,Monotonie der Beitragstabellen,dokumentintern,deterministisch,WARNUNG,0,1,"356 Altersübergänge geprüft. 1 Rückgang (höchstens 0,16 %) beim in der TB begründeten Übergang 64 → 65 (Begründung Seite 36) – plausibel, nicht gemeldet."
V-07,Abgleich mit Referenzwerten (CSV),dokumentextern,deterministisch,FEHLER,2,0,"24 Spalten den Referenz-CSVs zugeordnet, 2160 Zellen verglichen."
V-08,Konsistenz benannter Parameter,dokumentintern,deterministisch,FEHLER,1,0,14 Parameter-Zuweisungen zu 12 Symbolen geprüft.
V-09,Gesetzes- und Paragrafen-Prüfung,dokumentextern,deterministisch,FEHLER,3,0,6 Normbezüge bzw. Zuschlagsangaben geprüft.
V-10,Referenzen auf Rahmen-Berechnungsgrundlagen,dokumentextern,deterministisch,BESTANDEN,0,0,8 Referenzen gegen ein Register mit 3 Einträgen geprüft.


Check,Schwere,Seite,Befund
V-01,FEHLER,45,"Nicht aufgelöster Verweis: „..., ax+1/2 = 1 2 (︀ax+ ax+1 )︀, A111 := 0, a111 := 0, (32) verwendet (Anlage 5.2, Abschnitt ??); die Bilanzdeckungsrückstellung einer ...“"
V-03,FEHLER,3,"Titel-Abweichung bei Anlage 3.1: IHV nennt „Grundwerte der Versicherungstechnik“, die Überschrift lautet aber „Versicherungstechnische Grundwerte“ (Seite 81)."
V-04,FEHLER,7,"Symbol „DQx“ (Stamm „DQ“) wird 19-mal verwendet (u. a. Seiten [7, 20, 34, 44]), ist im Symbolverzeichnis aber nicht erklärt."
V-05,FEHLER,62,"Anlage 2.2 – Ausscheideordnung Männer: qx, wx, (q+ w)xund DQx…, Alter 50: (q+w)_x = 0,037271 stimmt nicht mit q_x + w_x = 0,036271 überein."
V-05,FEHLER,60,"Zins-Übersicht (Anlage 2.1, Seite 60) nennt für „F“ 2,2 %, die Tabelle Anlage 2.2 – Ausscheideordnung Frauen: qx, wx, (q+ w)xund DQx… ist aber erkennbar mit 2.0 % gerechnet (aus DQ_x = v·(1−(q+w)_x))."
V-06,WARNUNG,89,"Anhang 3, Spalte b_x (Spalte 3): Wert fällt von Alter 50 (653,33) auf Alter 51 (651,63). Die TB begründet an keiner Stelle einen Rückgang bei diesem Übergang. Bitte prüfen."
V-07,FEHLER,62,"Anlage 2.2 – Ausscheideordnung Männer: qx, wx, (q+ w)xund DQx…, Alter 50, Spalte „qw“ (Nr. 4): PDF zeigt 0,037271, Referenz (rechnungsgrundlagen_M.csv) ergibt 0,036271."
V-07,FEHLER,89,"Anhang 3, Alter 50, Spalte „b“ (Nr. 3): PDF zeigt 653,33, Referenz (rechnungsgrundlagen_M.csv) ergibt 635,33."
V-08,FEHLER,73,"Widersprüchliche Werte für „GM“: 1910,0070 EUR (Seite 73); 1901,0070 EUR (Seite 73)."
V-09,FEHLER,16,„Höchstrechnungszins“ wird mit „§ 4 VAG“ belegt – einschlägig ist jedoch § 4 KVAV. (Der Rechnungszins in der PKV ist in § 4 KVAV geregelt.)



GESAMTERGEBNIS: 21 Fehler, 11 Warnungen (19 Checks, davon 0 übersprungen)
=> Das Dokument enthält Befunde, die vor einer Freigabe zu klären sind.
Prüfbericht gespeichert: pruefbericht_TB_VitalPlus_fehlerhaft.json


## 10 | Grenzen und Erweiterung

**Eigene Checks ergänzen** – eine Funktion mit Dekorator genügt, danach die
Zellen 8–9 erneut ausführen:

```python
@check("V-11", "Mein neuer Check", "dokumentintern", "deterministisch")
def check_mein_thema(dok):
    befunde = []
    # ... dok.seiten, dok.tabellen, dok.ueberschriften, dok.paragrafen ...
    return befunde, "Beschreibung des Prüfumfangs"
```

**Bekannte Grenzen:**

- Die Tabellenerkennung setzt Alterstabellen (erste Spalte = Alter) mit
  eingebettetem Text-Layer voraus; gescannte PDFs ohne Text-Layer bräuchten
  vorgelagerte OCR.
- Der CSV-Abgleich (V-07) ordnet Spalten über kanonische Namen und
  Wertübereinstimmung zu; Referenzdaten mit abweichender Semantik bleiben
  unberücksichtigt (der Check weist die Zahl der zugeordneten Spalten aus).
- Die Wissensbasis in V-09 enthält bewusst nur gesicherte Norm-Zuordnungen;
  die breitere Prüfung übernimmt L-04 am amtlichen Wortlaut
  (gesetze-im-internet.de, ohne Internet aus `normtexte.json`). L-04 prüft,
  ob die zitierte Stelle den Sachverhalt regelt – nicht, ob die TB die
  Vorschrift inhaltlich einhält.
- **Formeln im Text-Layer**: Die Textextraktion zerlegt gesetzte Formeln
  in Fragmente (mehrzeilige Klammern, verlorene Faktoren) – textbasierte
  Checks können reine Druckfehler im Formelbild daher nicht sehen, wenn sie
  sich in keinem Zahlenwert niederschlagen. Genau dafür ergänzt L-05 die
  Vision-Prüfung: Die formelhaltigen Seiten werden als Bild geprüft, wodurch
  im Test auch das gedrehte γ-Vorzeichen der AR-Formel (F12) gefunden wurde.
  Die Vision-Prüfung bleibt eine LLM-Einschätzung – nicht eindeutige
  Hinweise erscheinen als Warnung.
- Der Grafik-Tabellen-Abgleich (L-06) kann Kurvenwerte nur mit der
  Genauigkeit der Bildauflösung und Achsenskala ablesen. Die Ablesung
  erfolgt bewusst **blind** (das Modell bekommt die Tabellenwerte nicht zu
  sehen): Ein Modell, das die Soll-Werte kennt, neigt dazu, sie schlicht zu
  „bestätigen“ (Verankerungseffekt) – im Test übersah es so eine um 50 %
  überhöhte Kurve. Der eigentliche Vergleich läuft deshalb deterministisch
  in Python (Zuordnung jeder Kurve zur am besten passenden Tabellenspalte,
  krass = Median-Abweichung über 20 %); feine Abweichungen unterhalb der
  Ablesegenauigkeit bleiben unerkannt, die Genauigkeit wird je Diagramm im
  Prüfumfang ausgewiesen.
- LLM-Checks sind nicht deterministisch; der Antwort-Cache friert einen
  Stand ein. Für einen frischen Lauf die Cache-Datei löschen.
- Ein bestandener Lauf ist **kein** Ersatz für die aktuarielle Durchsicht –
  das Notebook findet Inkonsistenzen, beurteilt aber keine Angemessenheit
  von Rechnungsgrundlagen.